<a href="https://colab.research.google.com/github/zaidalraggad314-coder/trump-tweets-oil-analysis/blob/main/trump_tweets_oil_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Trump Tweets EDA + NLP Sentiment × Oil Prediction — Corrected Notebook

This notebook fixes the critical issues from the previous version:

1. Fixes market/geopolitical keyword detection.
2. Removes noisy `pic.twitter.com` and URL artifacts before sentiment scoring.
3. Uses a stronger sentiment approach: **VADER when available**, with TextBlob fallback.
4. Separates **EDA**, **regression**, and **classification**.
5. Uses a **chronological train/test split**, not random splitting.
6. Blocks current-day oil leakage from the feature matrix.
7. Compares tweet/sentiment models against honest oil-only and dummy baselines.
8. Frames results realistically: this tests association and weak predictive signal, not causality.

> Project framing: This notebook tests whether Trump tweet signals are associated with short-term oil price movement. It does **not** claim that tweets alone control oil prices.

## 1) Setup
Run this cell first. If you are using Google Colab, upload the CSV files before running the notebook.

In [ ]:
import os
import re
import sys
import math
import warnings
from pathlib import Path
from collections import Counter

warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 160)
pd.set_option('display.max_colwidth', 240)

RANDOM_STATE = 42
TEST_SIZE = 0.20

## 2) Optional Plotly Dashboard Style
If Plotly is available, the notebook will create interactive professional charts. If not, it will continue with matplotlib.

In [ ]:
try:
    import plotly.express as px
    import plotly.graph_objects as go
    import plotly.io as pio
    PLOTLY_OK = True
except Exception:
    PLOTLY_OK = False
    px = None
    go = None
    pio = None

if PLOTLY_OK:
    pio.templates['oil_sentiment_dark_corrected'] = pio.templates['plotly_dark']
    pio.templates['oil_sentiment_dark_corrected'].layout.update(
        font=dict(family='Arial', size=14, color='#E5E7EB'),
        title=dict(font=dict(size=22, color='#F8FAFC'), x=0.5),
        paper_bgcolor='#0F172A',
        plot_bgcolor='#111827',
        colorway=['#38BDF8', '#FACC15', '#34D399', '#FB7185', '#A78BFA', '#F472B6', '#22D3EE', '#F97316'],
        xaxis=dict(gridcolor='#334155', zerolinecolor='#475569'),
        yaxis=dict(gridcolor='#334155', zerolinecolor='#475569'),
        legend=dict(bgcolor='rgba(15,23,42,0.65)', bordercolor='#334155', borderwidth=1)
    )
    pio.templates.default = 'oil_sentiment_dark_corrected'

def style_fig(fig, title=None, height=520):
    if not PLOTLY_OK:
        return fig
    fig.update_layout(
        template='oil_sentiment_dark_corrected',
        title=title or fig.layout.title.text,
        title_x=0.5,
        height=height,
        margin=dict(l=40, r=40, t=75, b=45),
        hovermode='x unified'
    )
    return fig

## 3) File Paths
The notebook looks for the expected files in Colab, local folders, and `/mnt/data`.

Expected files:
- `tweets2017-2026.csv`
- `Crude_Oil_daily.csv`
- Optional: `oil_hourly_2017.csv`

In [ ]:
TWEETS_PATH_CANDIDATES = [
    '/content/tweets2017-2026.csv',
    '/content/tweets2017-2026(2).csv',
    'tweets2017-2026.csv',
    'tweets2017-2026(2).csv',
    '/mnt/data/tweets2017-2026.csv',
]

OIL_DAILY_PATH_CANDIDATES = [
    '/content/Crude_Oil_daily.csv',
    '/content/Crude_Oil_daily(2).csv',
    'Crude_Oil_daily.csv',
    'Crude_Oil_daily(2).csv',
    '/mnt/data/Crude_Oil_daily.csv',
]

OIL_HOURLY_PATH_CANDIDATES = [
    '/content/oil_hourly_2017.csv',
    '/content/oil_hourly_2017(2).csv',
    'oil_hourly_2017.csv',
    'oil_hourly_2017(2).csv',
    '/mnt/data/oil_hourly_2017.csv',
]

def first_existing(paths, required=True):
    for p in paths:
        if os.path.exists(p):
            return p
    if required:
        raise FileNotFoundError(f'None of these files were found: {paths}')
    return None

TWEETS_PATH = first_existing(TWEETS_PATH_CANDIDATES, required=True)
OIL_DAILY_PATH = first_existing(OIL_DAILY_PATH_CANDIDATES, required=True)
OIL_HOURLY_PATH = first_existing(OIL_HOURLY_PATH_CANDIDATES, required=False)

print('Tweets file:', TWEETS_PATH)
print('Daily oil file:', OIL_DAILY_PATH)
print('Hourly oil file:', OIL_HOURLY_PATH)

Tweets file: /content/tweets2017-2026.csv
Daily oil file: /content/Crude_Oil_daily.csv
Hourly oil file: /content/oil_hourly_2017.csv


## 4) Load Raw Data
This cell only loads the files. Cleaning is done in later sections.

In [ ]:
df_raw = pd.read_csv(TWEETS_PATH)
oil_daily_raw = pd.read_csv(OIL_DAILY_PATH)

oil_hourly_raw = None
if OIL_HOURLY_PATH is not None:
    oil_hourly_raw = pd.read_csv(OIL_HOURLY_PATH)

print('Tweets shape:', df_raw.shape)
print('Daily oil shape:', oil_daily_raw.shape)
if oil_hourly_raw is not None:
    print('Hourly oil shape:', oil_hourly_raw.shape)

display(df_raw.head())
display(oil_daily_raw.head())
if oil_hourly_raw is not None:
    display(oil_hourly_raw.head())

Tweets shape: (58392, 18)
Daily oil shape: (2252, 8)
Hourly oil shape: (89395, 7)


,id,date,platform,handle,text,favorite_count,repost_count,quote_flag,repost_flag,deleted_flag,word_count,hashtags,urls,user_mentions,media_count,media_urls,post_url,in_reply_to
0,8.220000e+17,2017-01-20 00:40:51+00:00,Twitter,realDonaldTrump,"Thank you for joining us at the Lincoln Memorial tonight- a very special evening! Together, we are going to MAKE AMERICA GREAT AGAIN! pic.twitter.com/5d774OCx5o",146346,28852,False,False,False,24,NaN,https://twitter.com/i/web/status/822242449053614081,NaN,0,NaN,https://x.com/realdonaldtrump/status/822242449053614081,NaN
1,8.220000e+17,2017-01-20 04:24:33+00:00,Twitter,realDonaldTrump,"Thank you for a wonderful evening in Washington, D.C. #Inauguration pic.twitter.com/a6xpFQTHj5",98164,17221,False,False,False,11,#inauguration,NaN,NaN,1,https://pbs.twimg.com/media/C2lkWIQUUAAQJVH.jpg,https://x.com/realdonaldtrump/status/822298747421986828,NaN
2,8.220000e+17,2017-01-20 12:31:53+00:00,Twitter,realDonaldTrump,It all begins today! I will see you at 11:00 A.M. for the swearing-in. THE MOVEMENT CONTINUES - THE WORK BEGINS!,234807,58312,False,False,False,21,NaN,NaN,NaN,0,NaN,https://x.com/realdonaldtrump/status/822421390125043713,NaN
3,8.230000e+17,2017-01-20 17:51:25+00:00,Twitter,realDonaldTrump,"Today we are not merely transferring power from one Administration to another, or from one party to another – but we are transferring...",95763,16646,False,False,False,23,NaN,NaN,NaN,0,NaN,https://x.com/realdonaldtrump/status/822501803615014918,NaN
4,8.230000e+17,2017-01-20 17:51:58+00:00,Twitter,realDonaldTrump,"power from Washington, D.C. and giving it back to you, the American People. #InaugurationDay",79061,15074,False,False,False,14,#inaugurationday,NaN,NaN,0,NaN,https://x.com/realdonaldtrump/status/822501939267141634,realDonaldTrump


,Date,Open,High,Low,Close,Volume,ticker,name
0,2017-01-20 00:00:00-05:00,51.450001,52.900002,51.389999,52.419998,567231,CL=F,Crude Oil Futures (CL=F)
1,2017-01-23 00:00:00-05:00,53.330002,53.470001,52.209999,52.750000,455333,CL=F,Crude Oil Futures (CL=F)
2,2017-01-24 00:00:00-05:00,52.860001,53.560001,52.669998,53.180000,520285,CL=F,Crude Oil Futures (CL=F)
3,2017-01-25 00:00:00-05:00,52.950001,53.470001,52.560001,52.750000,589709,CL=F,Crude Oil Futures (CL=F)
4,2017-01-26 00:00:00-05:00,52.959999,54.060001,52.790001,53.779999,578065,CL=F,Crude Oil Futures (CL=F)


,datetime,open,high,low,close,volume,oil_type
0,1/20/2017 0:00,54.42,54.43,54.33,54.38,0,Brent
1,1/20/2017 0:00,52.30,52.31,52.24,52.29,0,WTI
2,1/20/2017 1:00,54.37,54.37,54.19,54.21,0,Brent
3,1/20/2017 1:00,52.28,52.28,52.14,52.16,0,WTI
4,1/20/2017 2:00,54.20,54.45,54.16,54.44,0,Brent


## 5) Basic Column Validation
This prevents silent failure when expected columns are missing.

In [ ]:
required_tweet_cols = ['date', 'text']
missing_tweet_cols = [c for c in required_tweet_cols if c not in df_raw.columns]
if missing_tweet_cols:
    raise ValueError(f'Missing required tweet columns: {missing_tweet_cols}')

print('Tweet columns available:')
print(df_raw.columns.tolist())
print('\nOil daily columns available:')
print(oil_daily_raw.columns.tolist())

Tweet columns available:
['id', 'date', 'platform', 'handle', 'text', 'favorite_count', 'repost_count', 'quote_flag', 'repost_flag', 'deleted_flag', 'word_count', 'hashtags', 'urls', 'user_mentions', 'media_count', 'media_urls', 'post_url', 'in_reply_to']

Oil daily columns available:
['Date', 'Open', 'High', 'Low', 'Close', 'Volume', 'ticker', 'name']


## 6) Clean Tweet Data
Main fixes:
- Do not drop rows blindly.
- Keep rows with valid `text` and `date`.
- Convert engagement columns safely.
- Standardize date columns in UTC.

In [ ]:
df = df_raw.copy()

df = df.dropna(subset=['text']).copy()
df['date'] = pd.to_datetime(df['date'], errors='coerce', utc=True)
df = df.dropna(subset=['date']).copy()

optional_text_cols = ['hashtags', 'urls', 'user_mentions', 'in_reply_to', 'platform', 'handle']
for col in optional_text_cols:
    if col not in df.columns:
        df[col] = ''
    df[col] = df[col].fillna('').astype(str)

for col in ['favorite_count', 'repost_count', 'word_count', 'media_count']:
    if col not in df.columns:
        df[col] = 0
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

if 'id' not in df.columns:
    df['id'] = np.arange(len(df))

# Time features
for col in ['year', 'month', 'day', 'dayofweek', 'dayofyear', 'quarter']:
    df[col] = getattr(df['date'].dt, col)

df['weekofyear'] = df['date'].dt.isocalendar().week.astype(int)
df['date_day'] = df['date'].dt.floor('D')
df['date_hour'] = df['date'].dt.floor('H')

df['tweet_length_ch'] = df['text'].astype(str).str.len()
df['tweet_length_class'] = pd.cut(
    df['tweet_length_ch'],
    bins=[-1, 130, 280, np.inf],
    labels=['short', 'medium', 'long']
)

df['engagement'] = df['favorite_count'] + df['repost_count']

def count_hashtags(row):
    h_col = str(row.get('hashtags', ''))
    h_text = str(row.get('text', ''))
    return max(h_col.count('#'), h_text.count('#'))

def count_mentions(row):
    m_col = str(row.get('user_mentions', ''))
    m_text = str(row.get('text', ''))
    return max(m_col.count('@'), m_text.count('@'))

df['hashtag_count'] = df.apply(count_hashtags, axis=1)
df['mention_count'] = df.apply(count_mentions, axis=1)
df['media_count_clean'] = df['media_count'].fillna(0).astype(int)

print('Clean tweets shape:', df.shape)
display(df[['date', 'text', 'platform', 'favorite_count', 'repost_count', 'engagement']].head())

Clean tweets shape: (58389, 33)


,date,text,platform,favorite_count,repost_count,engagement
0,2017-01-20 00:40:51+00:00,"Thank you for joining us at the Lincoln Memorial tonight- a very special evening! Together, we are going to MAKE AMERICA GREAT AGAIN! pic.twitter.com/5d774OCx5o",Twitter,146346,28852,175198
1,2017-01-20 04:24:33+00:00,"Thank you for a wonderful evening in Washington, D.C. #Inauguration pic.twitter.com/a6xpFQTHj5",Twitter,98164,17221,115385
2,2017-01-20 12:31:53+00:00,It all begins today! I will see you at 11:00 A.M. for the swearing-in. THE MOVEMENT CONTINUES - THE WORK BEGINS!,Twitter,234807,58312,293119
3,2017-01-20 17:51:25+00:00,"Today we are not merely transferring power from one Administration to another, or from one party to another – but we are transferring...",Twitter,95763,16646,112409
4,2017-01-20 17:51:58+00:00,"power from Washington, D.C. and giving it back to you, the American People. #InaugurationDay",Twitter,79061,15074,94135


## 7) Stronger Text Cleaning for NLP
Critical fix: remove noisy URL artifacts such as `pic.twitter.com/...`, even when the link appears without `https://`.

In [ ]:
def clean_text_for_sentiment(text):
    x = str(text)
    x = re.sub(r'https?://\S+', ' ', x, flags=re.IGNORECASE)
    x = re.sub(r'www\.\S+', ' ', x, flags=re.IGNORECASE)
    x = re.sub(r'pic\.twitter\.com/\S+', ' ', x, flags=re.IGNORECASE)
    x = re.sub(r't\.co/\S+', ' ', x, flags=re.IGNORECASE)
    x = re.sub(r'@\w+', ' ', x)
    x = re.sub(r'#', '', x)
    x = re.sub(r'[^a-zA-Z\s]', ' ', x)
    x = re.sub(r'\s+', ' ', x).strip().lower()
    return x

df['clean_text'] = df['text'].apply(clean_text_for_sentiment)

display(df[['text', 'clean_text']].head())

,text,clean_text
0,"Thank you for joining us at the Lincoln Memorial tonight- a very special evening! Together, we are going to MAKE AMERICA GREAT AGAIN! pic.twitter.com/5d774OCx5o",thank you for joining us at the lincoln memorial tonight a very special evening together we are going to make america great again
1,"Thank you for a wonderful evening in Washington, D.C. #Inauguration pic.twitter.com/a6xpFQTHj5",thank you for a wonderful evening in washington d c inauguration
2,It all begins today! I will see you at 11:00 A.M. for the swearing-in. THE MOVEMENT CONTINUES - THE WORK BEGINS!,it all begins today i will see you at a m for the swearing in the movement continues the work begins
3,"Today we are not merely transferring power from one Administration to another, or from one party to another – but we are transferring...",today we are not merely transferring power from one administration to another or from one party to another but we are transferring
4,"power from Washington, D.C. and giving it back to you, the American People. #InaugurationDay",power from washington d c and giving it back to you the american people inaugurationday


## 8) Sentiment Scoring — VADER First, TextBlob Fallback
VADER is usually better for short social media text. If VADER is not available, the notebook falls back to TextBlob.

In [ ]:
SENTIMENT_ENGINE = None

try:
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
    vader_analyzer = SentimentIntensityAnalyzer()
    SENTIMENT_ENGINE = 'VADER'
except Exception:
    try:
        import subprocess
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'vaderSentiment', '-q'])
        from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
        vader_analyzer = SentimentIntensityAnalyzer()
        SENTIMENT_ENGINE = 'VADER'
    except Exception:
        vader_analyzer = None
        SENTIMENT_ENGINE = 'TextBlob'

if SENTIMENT_ENGINE == 'TextBlob':
    try:
        from textblob import TextBlob
    except Exception:
        import subprocess
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'textblob', '-q'])
        from textblob import TextBlob

print('Sentiment engine:', SENTIMENT_ENGINE)

Sentiment engine: VADER


In [ ]:
def sentiment_scores(text):
    text = str(text)
    if SENTIMENT_ENGINE == 'VADER':
        scores = vader_analyzer.polarity_scores(text)
        return pd.Series({
            'polarity': scores['compound'],
            'positive_score': scores['pos'],
            'neutral_score': scores['neu'],
            'negative_score': scores['neg'],
            'subjectivity': np.nan
        })
    else:
        blob = TextBlob(text)
        return pd.Series({
            'polarity': blob.sentiment.polarity,
            'positive_score': max(blob.sentiment.polarity, 0),
            'neutral_score': 1 - abs(blob.sentiment.polarity),
            'negative_score': abs(min(blob.sentiment.polarity, 0)),
            'subjectivity': blob.sentiment.subjectivity
        })

def sentiment_label(score):
    if score > 0.05:
        return 'Positive'
    elif score < -0.05:
        return 'Negative'
    return 'Neutral'

sentiment_df = df['clean_text'].apply(sentiment_scores)
df = pd.concat([df, sentiment_df], axis=1)
df['subjectivity'] = df['subjectivity'].fillna(0)
df['sentiment'] = df['polarity'].apply(sentiment_label)
df['sentiment_score'] = df['sentiment'].map({'Positive': 1, 'Neutral': 0, 'Negative': -1})

print(df['sentiment'].value_counts())
display(df[['text', 'clean_text', 'polarity', 'sentiment']].head())

sentiment
Positive    24735
Neutral     18491
Negative    15163
Name: count, dtype: int64


,text,clean_text,polarity,sentiment
0,"Thank you for joining us at the Lincoln Memorial tonight- a very special evening! Together, we are going to MAKE AMERICA GREAT AGAIN! pic.twitter.com/5d774OCx5o",thank you for joining us at the lincoln memorial tonight a very special evening together we are going to make america great again,0.8622,Positive
1,"Thank you for a wonderful evening in Washington, D.C. #Inauguration pic.twitter.com/a6xpFQTHj5",thank you for a wonderful evening in washington d c inauguration,0.7351,Positive
2,It all begins today! I will see you at 11:00 A.M. for the swearing-in. THE MOVEMENT CONTINUES - THE WORK BEGINS!,it all begins today i will see you at a m for the swearing in the movement continues the work begins,-0.2500,Negative
3,"Today we are not merely transferring power from one Administration to another, or from one party to another – but we are transferring...",today we are not merely transferring power from one administration to another or from one party to another but we are transferring,0.2144,Positive
4,"power from Washington, D.C. and giving it back to you, the American People. #InaugurationDay",power from washington d c and giving it back to you the american people inaugurationday,0.3400,Positive


## 9) Correct Market and Geopolitical Keyword Detection
This fixes the broken issue where `Market-Related Posts` was returning zero.

Design:
- Use word boundaries so `gas` does not match `vegas`.
- Keep phrases like `supply chain`, `interest rate`, and `crude oil`.
- Validate matches immediately after creating the feature.

In [ ]:
market_keywords = [
    'oil', 'gas', 'energy', 'crude', 'crude oil', 'petroleum', 'opec',
    'saudi', 'iran', 'iraq', 'russia', 'china', 'war', 'missile',
    'sanction', 'sanctions', 'tariff', 'tariffs', 'trade', 'supply', 'supply chain',
    'shipping', 'inflation', 'prices', 'market', 'economy', 'economic',
    'jobs', 'dollar', 'fed', 'interest', 'interest rate', 'rates', 'stock', 'stocks',
    'exports', 'imports', 'manufacturing', 'pipeline', 'refinery', 'fuel'
]

geopolitical_keywords = [
    'iran', 'iraq', 'russia', 'china', 'saudi', 'opec', 'war', 'missile', 'military',
    'sanction', 'sanctions', 'tariff', 'tariffs', 'trade', 'border', 'conflict',
    'ukraine', 'nato', 'north korea', 'israel', 'gaza', 'red sea', 'houthi'
]

# Stronger subset focused directly on oil/energy.
oil_direct_keywords = [
    'oil', 'crude', 'crude oil', 'gas', 'energy', 'petroleum', 'opec', 'fuel',
    'pipeline', 'refinery', 'drilling', 'shale'
]

def compile_keyword_patterns(keywords):
    patterns = []
    for kw in sorted(set(keywords), key=len, reverse=True):
        escaped = re.escape(kw.lower()).replace('\\ ', r'\s+')
        pattern = re.compile(r'(?<![a-zA-Z])' + escaped + r'(?![a-zA-Z])', flags=re.IGNORECASE)
        patterns.append((kw, pattern))
    return patterns

market_patterns = compile_keyword_patterns(market_keywords)
geo_patterns = compile_keyword_patterns(geopolitical_keywords)
oil_direct_patterns = compile_keyword_patterns(oil_direct_keywords)

def keyword_count(text, patterns):
    text = str(text).lower()
    return sum(1 for _, pattern in patterns if pattern.search(text))

def matched_keywords(text, patterns):
    text = str(text).lower()
    return [kw for kw, pattern in patterns if pattern.search(text)]

df['market_keyword_count'] = df['clean_text'].apply(lambda x: keyword_count(x, market_patterns))
df['geo_keyword_count'] = df['clean_text'].apply(lambda x: keyword_count(x, geo_patterns))
df['oil_direct_keyword_count'] = df['clean_text'].apply(lambda x: keyword_count(x, oil_direct_patterns))

df['market_keywords_matched'] = df['clean_text'].apply(lambda x: ', '.join(matched_keywords(x, market_patterns)))
df['geo_keywords_matched'] = df['clean_text'].apply(lambda x: ', '.join(matched_keywords(x, geo_patterns)))

df['is_market_related'] = (df['market_keyword_count'] > 0).astype(int)
df['is_geo_related'] = (df['geo_keyword_count'] > 0).astype(int)
df['is_oil_direct_related'] = (df['oil_direct_keyword_count'] > 0).astype(int)

df['abs_polarity'] = df['polarity'].abs()
df['log_engagement'] = np.log1p(df['engagement'])

df['final_impact_sentiment'] = df['polarity'] * (1 + df['market_keyword_count']) * df['log_engagement']
df['market_importance'] = df['is_market_related'] * (1 + df['geo_keyword_count']) * df['log_engagement']
df['oil_direct_importance'] = df['is_oil_direct_related'] * (1 + df['geo_keyword_count']) * df['log_engagement']

summary_keyword_check = pd.DataFrame({
    'Metric': [
        'Market-related posts',
        'Geo-related posts',
        'Direct oil/energy posts',
        'Total market keyword mentions',
        'Total geo keyword mentions',
        'Total direct oil keyword mentions'
    ],
    'Value': [
        int(df['is_market_related'].sum()),
        int(df['is_geo_related'].sum()),
        int(df['is_oil_direct_related'].sum()),
        int(df['market_keyword_count'].sum()),
        int(df['geo_keyword_count'].sum()),
        int(df['oil_direct_keyword_count'].sum()),
    ]
})

display(summary_keyword_check)

if df['is_market_related'].sum() == 0:
    raise ValueError('Market keyword detection failed: zero market-related posts found. Check regex/text cleaning.')

,Metric,Value
0,Market-related posts,6734
1,Geo-related posts,6646
2,Direct oil/energy posts,1077
3,Total market keyword mentions,10033
4,Total geo keyword mentions,8977
5,Total direct oil keyword mentions,1224


## 9.1) Added Improvement — Expanded Keyword Coverage
This cell extends the existing keyword system without removing the original logic. It adds more energy-market, macroeconomic, and geopolitical terms, then recalculates the keyword-based features used later in the notebook.


In [ ]:
# =========================
# Added Improvement: Expanded Energy + Geopolitical Keyword System
# Place: directly after the original keyword detection cell
# =========================

energy_keywords_extra = [
    # Energy and oil market terms
    'brent', 'wti', 'crude', 'crude oil', 'oil prices', 'oil price',
    'gasoline', 'diesel', 'natural gas', 'lng', 'energy prices',
    'oil supply', 'oil demand', 'supply cut', 'production cut',
    'output cut', 'barrel', 'barrels', 'shale', 'drilling',
    'refinery', 'refineries', 'pipeline', 'pipelines',
    'strategic petroleum reserve', 'spr', 'opec+', 'opec plus'
]

geopolitical_keywords_extra = [
    # Geopolitical and supply-chain risk terms
    'middle east', 'persian gulf', 'strait of hormuz', 'red sea',
    'suez canal', 'yemen', 'houthi', 'houthis', 'gulf',
    'sanctions', 'embargo', 'tariff', 'tariffs',
    'trade war', 'conflict', 'military strike', 'missile attack',
    'ukraine', 'nato', 'russia', 'iran', 'saudi arabia',
    'china', 'israel', 'gaza'
]

macro_keywords_extra = [
    # Economic words that may affect oil indirectly
    'inflation', 'interest rate', 'rates', 'fed', 'federal reserve',
    'recession', 'growth', 'gdp', 'dollar', 'usd',
    'jobs report', 'unemployment', 'manufacturing',
    'trade deficit', 'exports', 'imports'
]

# Keep the original lists and only expand them.
market_keywords = sorted(set(market_keywords + energy_keywords_extra + geopolitical_keywords_extra + macro_keywords_extra))
geopolitical_keywords = sorted(set(geopolitical_keywords + geopolitical_keywords_extra))
oil_direct_keywords = sorted(set(oil_direct_keywords + energy_keywords_extra))

# Rebuild patterns using the original helper functions.
market_patterns = compile_keyword_patterns(market_keywords)
geo_patterns = compile_keyword_patterns(geopolitical_keywords)
oil_direct_patterns = compile_keyword_patterns(oil_direct_keywords)

# Recalculate keyword features using the expanded lists.
df['market_keyword_count'] = df['clean_text'].apply(lambda x: keyword_count(x, market_patterns))
df['geo_keyword_count'] = df['clean_text'].apply(lambda x: keyword_count(x, geo_patterns))
df['oil_direct_keyword_count'] = df['clean_text'].apply(lambda x: keyword_count(x, oil_direct_patterns))

df['market_keywords_matched'] = df['clean_text'].apply(lambda x: ', '.join(matched_keywords(x, market_patterns)))
df['geo_keywords_matched'] = df['clean_text'].apply(lambda x: ', '.join(matched_keywords(x, geo_patterns)))
df['oil_keywords_matched'] = df['clean_text'].apply(lambda x: ', '.join(matched_keywords(x, oil_direct_patterns)))

df['is_market_related'] = (df['market_keyword_count'] > 0).astype(int)
df['is_geo_related'] = (df['geo_keyword_count'] > 0).astype(int)
df['is_oil_direct_related'] = (df['oil_direct_keyword_count'] > 0).astype(int)

# Recalculate impact features so later daily aggregation uses the improved keyword detection.
df['final_impact_sentiment'] = df['polarity'] * (1 + df['market_keyword_count']) * df['log_engagement']
df['market_importance'] = df['is_market_related'] * (1 + df['geo_keyword_count']) * df['log_engagement']
df['oil_direct_importance'] = df['is_oil_direct_related'] * (1 + df['geo_keyword_count']) * df['log_engagement']

summary_keyword_check = pd.DataFrame({
    'Metric': [
        'Market-related posts',
        'Geo-related posts',
        'Direct oil/energy posts',
        'Total market keyword mentions',
        'Total geo keyword mentions',
        'Total direct oil keyword mentions'
    ],
    'Value': [
        int(df['is_market_related'].sum()),
        int(df['is_geo_related'].sum()),
        int(df['is_oil_direct_related'].sum()),
        int(df['market_keyword_count'].sum()),
        int(df['geo_keyword_count'].sum()),
        int(df['oil_direct_keyword_count'].sum()),
    ]
})

print('Expanded keyword system completed.')
print('Market keywords:', len(market_keywords))
print('Geopolitical keywords:', len(geopolitical_keywords))
print('Oil-direct keywords:', len(oil_direct_keywords))
display(summary_keyword_check)

display(
    df[['date', 'text', 'sentiment', 'market_keyword_count',
        'geo_keyword_count', 'oil_direct_keyword_count',
        'oil_keywords_matched']].head(10)
)


## 10) Inspect Market-Related Tweets
This cell verifies that the keyword feature is not silently broken.

In [ ]:
market_examples = df[df['is_market_related'] == 1][
    ['date', 'platform', 'text', 'market_keywords_matched', 'geo_keywords_matched', 'polarity', 'sentiment']
].head(15)

display(market_examples)

,date,platform,text,market_keywords_matched,geo_keywords_matched,polarity,sentiment
8,2017-01-20 17:54:36+00:00,Twitter,We will bring back our jobs. We will bring back our borders. We will bring back our wealth - and we will bring back our dreams!,jobs,,0.7096,Positive
22,2017-01-23 11:38:16+00:00,Twitter,Busy week planned with a heavy focus on jobs and national security. Top executives coming in at 9:00 A.M. to talk manufacturing in America.,"manufacturing, jobs",,0.4939,Positive
23,2017-01-24 11:11:47+00:00,Twitter,Will be meeting at 9:00 with top automobile executives concerning jobs in America. I want new plants to be built here for cars sold here!,jobs,,0.2732,Positive
41,2017-01-26 13:51:46+00:00,Twitter,The U.S. has a 60 billion dollar trade deficit with Mexico. It has been a one-sided deal from the beginning of NAFTA with massive numbers...,"dollar, trade",trade,-0.4019,Negative
42,2017-01-26 13:55:03+00:00,Twitter,"of jobs and companies lost. If Mexico is unwilling to pay for the badly needed wall, then it would be better to cancel the upcoming meeting.",jobs,,-0.5994,Negative
48,2017-01-27 13:19:10+00:00,Twitter,"Mexico has taken advantage of the U.S. for long enough. Massive trade deficits & little help on the very weak border must change, NOW!",trade,"border, trade",0.0552,Positive
55,2017-01-28 13:08:42+00:00,Twitter,Thr coverage about me in the @nytimes and the @washingtonpost gas been so false and angry that the times actually apologized to its.....,gas,,-0.3102,Negative
63,2017-01-29 21:49:32+00:00,Twitter,"...Senators should focus their energies on ISIS, illegal immigration and border security instead of always looking to start World War III.",war,"border, war",-0.6369,Negative
71,2017-01-30 14:23:49+00:00,Twitter,Where was all the outrage from Democrats and the opposition party (the media) when our jobs were fleeing our country?,jobs,,-0.1531,Negative
80,2017-02-02 03:06:14+00:00,Twitter,Iran is rapidly taking over more and more of Iraq even after the U.S. has squandered three trillion dollars there. Obvious long ago!,"iraq, iran","iraq, iran",0.0000,Neutral


## 11) EDA Overview
EDA can use all available historical data. Modeling evaluation will use only the chronological test period.

In [ ]:
summary_cards = pd.DataFrame({
    'Metric': [
        'Total Posts', 'Start Date', 'End Date', 'Sentiment Engine',
        'Average Polarity', 'Market-Related Posts', 'Direct Oil/Energy Posts'
    ],
    'Value': [
        len(df),
        str(df['date'].min()),
        str(df['date'].max()),
        SENTIMENT_ENGINE,
        round(df['polarity'].mean(), 4),
        int(df['is_market_related'].sum()),
        int(df['is_oil_direct_related'].sum())
    ]
})

display(summary_cards)

,Metric,Value
0,Total Posts,58389
1,Start Date,2017-01-20 00:40:51+00:00
2,End Date,2025-12-31 21:54:21+00:00
3,Sentiment Engine,VADER
4,Average Polarity,0.114
5,Market-Related Posts,6734
6,Direct Oil/Energy Posts,1077


In [ ]:
if PLOTLY_OK:
    sentiment_counts = df['sentiment'].value_counts().reset_index()
    sentiment_counts.columns = ['sentiment', 'count']
    fig = px.pie(sentiment_counts, names='sentiment', values='count', title='Sentiment Distribution')
    style_fig(fig, height=450).show()
else:
    df['sentiment'].value_counts().plot(kind='pie', autopct='%1.1f%%', figsize=(6,6), title='Sentiment Distribution')
    plt.ylabel('')
    plt.show()

In [ ]:
posts_by_year = df.groupby('year', as_index=False).agg(posts=('id', 'count'))

if PLOTLY_OK:
    fig = px.line(posts_by_year, x='year', y='posts', markers=True, title='Posts by Year')
    style_fig(fig).show()
else:
    posts_by_year.plot(x='year', y='posts', marker='o', title='Posts by Year')
    plt.show()

In [ ]:
platform_counts = df['platform'].replace('', 'Unknown').value_counts().head(10).reset_index()
platform_counts.columns = ['platform', 'posts']

if PLOTLY_OK:
    fig = px.bar(platform_counts, x='platform', y='posts', title='Top Platforms')
    style_fig(fig).show()
else:
    platform_counts.plot(kind='bar', x='platform', y='posts', title='Top Platforms')
    plt.show()

In [ ]:
market_by_year = df.groupby('year', as_index=False).agg(
    total_posts=('id', 'count'),
    market_posts=('is_market_related', 'sum'),
    geo_posts=('is_geo_related', 'sum'),
    oil_direct_posts=('is_oil_direct_related', 'sum')
)
market_by_year['market_share'] = market_by_year['market_posts'] / market_by_year['total_posts']

display(market_by_year)

if PLOTLY_OK:
    melted = market_by_year.melt(
        id_vars='year',
        value_vars=['market_posts', 'geo_posts', 'oil_direct_posts'],
        var_name='Type',
        value_name='Posts'
    )
    fig = px.line(melted, x='year', y='Posts', color='Type', markers=True, title='Market/Geopolitical/Oil-Related Posts by Year')
    style_fig(fig).show()

,year,total_posts,market_posts,geo_posts,oil_direct_posts,market_share
0,2017,2472,377,302,18,0.152508
1,2018,3579,666,796,39,0.186085
2,2019,7841,1109,1210,86,0.141436
3,2020,12249,1140,920,85,0.093069
4,2021,156,5,3,0,0.032051
5,2022,4171,333,306,97,0.079837
6,2023,10086,725,751,105,0.071882
7,2024,11191,1051,1135,224,0.093915
8,2025,6644,1328,1223,423,0.199880


## 12) Top Hashtags
This is descriptive only and does not enter the model unless explicitly engineered.

In [ ]:
all_hashtags = []
for text, hashtags in zip(df['text'], df['hashtags']):
    found = re.findall(r'#\w+', str(text).lower())
    if not found:
        found = re.findall(r'#\w+', str(hashtags).lower())
    all_hashtags.extend(found)

top_hashtags = pd.DataFrame(Counter(all_hashtags).most_common(20), columns=['hashtag', 'count'])
display(top_hashtags)

if len(top_hashtags) > 0 and PLOTLY_OK:
    fig = px.bar(top_hashtags, x='count', y='hashtag', orientation='h', title='Top Hashtags')
    style_fig(fig).show()

,hashtag,count
0,#maga,551
1,#trump2024,94
2,#1,85
3,#kag2020,77
4,#covid19,67
5,#americafirst,56
6,#truth,47
7,#trumprally,46
8,#usa,45
9,#usmca,42


## 13) Prepare Daily Tweet Features
All tweet signals are aggregated by day.

Important modeling assumption:
- Features from day **t** are used to predict oil movement on the **next trading day**.
- This is a daily forecasting setup, not intraday trading proof.

In [ ]:
daily_tweets = df.groupby('date_day').agg(
    avg_sentiment=('sentiment_score', 'mean'),
    avg_polarity=('polarity', 'mean'),
    avg_subjectivity=('subjectivity', 'mean'),
    avg_positive_score=('positive_score', 'mean'),
    avg_neutral_score=('neutral_score', 'mean'),
    avg_negative_score=('negative_score', 'mean'),
    tweet_count=('text', 'count'),
    negative_tweets=('sentiment', lambda x: (x == 'Negative').sum()),
    positive_tweets=('sentiment', lambda x: (x == 'Positive').sum()),
    neutral_tweets=('sentiment', lambda x: (x == 'Neutral').sum()),
    total_likes=('favorite_count', 'sum'),
    total_reposts=('repost_count', 'sum'),
    total_engagement=('engagement', 'sum'),
    avg_word_count=('word_count', 'mean'),
    hashtag_count=('hashtag_count', 'sum'),
    mention_count=('mention_count', 'sum'),
    market_keyword_mentions=('market_keyword_count', 'sum'),
    geo_keyword_mentions=('geo_keyword_count', 'sum'),
    oil_direct_keyword_mentions=('oil_direct_keyword_count', 'sum'),
    market_related_posts=('is_market_related', 'sum'),
    geo_related_posts=('is_geo_related', 'sum'),
    oil_direct_related_posts=('is_oil_direct_related', 'sum'),
    avg_final_impact=('final_impact_sentiment', 'mean'),
    total_final_impact=('final_impact_sentiment', 'sum'),
    avg_market_importance=('market_importance', 'mean'),
    total_market_importance=('market_importance', 'sum'),
    avg_oil_direct_importance=('oil_direct_importance', 'mean'),
    total_oil_direct_importance=('oil_direct_importance', 'sum'),
).reset_index().rename(columns={'date_day': 'day'})

ratio_specs = [
    ('negative_tweets', 'tweet_count', 'negative_ratio'),
    ('positive_tweets', 'tweet_count', 'positive_ratio'),
    ('neutral_tweets', 'tweet_count', 'neutral_ratio'),
    ('market_related_posts', 'tweet_count', 'market_related_ratio'),
    ('geo_related_posts', 'tweet_count', 'geo_related_ratio'),
    ('oil_direct_related_posts', 'tweet_count', 'oil_direct_related_ratio'),
]

for numerator, denominator, new_col in ratio_specs:
    daily_tweets[new_col] = daily_tweets[numerator] / daily_tweets[denominator].replace(0, np.nan)

daily_tweets = daily_tweets.fillna(0)

print('Daily tweet features:', daily_tweets.shape)
display(daily_tweets.head())

Daily tweet features: (2781, 35)


,day,avg_sentiment,avg_polarity,avg_subjectivity,avg_positive_score,avg_neutral_score,avg_negative_score,tweet_count,negative_tweets,positive_tweets,neutral_tweets,total_likes,total_reposts,total_engagement,avg_word_count,hashtag_count,mention_count,market_keyword_mentions,geo_keyword_mentions,oil_direct_keyword_mentions,market_related_posts,geo_related_posts,oil_direct_related_posts,avg_final_impact,total_final_impact,avg_market_importance,total_market_importance,avg_oil_direct_importance,total_oil_direct_importance,negative_ratio,positive_ratio,neutral_ratio,market_related_ratio,geo_related_ratio,oil_direct_related_ratio
0,2017-01-20 00:00:00+00:00,0.307692,0.161100,0.0,0.100615,0.868538,0.030846,13,2,6,5,1638144,350907,1989051,16.846154,5,0,1,0,0,1,0,0,2.557080,33.242045,0.931790,12.113272,0.0,0.0,0.153846,0.461538,0.384615,0.076923,0.0,0.0
1,2017-01-21 00:00:00+00:00,1.000000,0.860500,0.0,0.364500,0.635500,0.000000,4,0,4,0,644697,116191,760888,19.250000,0,1,0,0,0,0,0,0,10.279675,41.118699,0.000000,0.000000,0.0,0.0,0.000000,1.000000,0.000000,0.000000,0.0,0.0
2,2017-01-22 00:00:00+00:00,0.600000,0.369620,0.0,0.255000,0.634400,0.110400,5,1,4,0,962338,187045,1149383,23.400000,0,1,0,0,0,0,0,0,4.492435,22.462177,0.000000,0.000000,0.0,0.0,0.200000,0.800000,0.000000,0.000000,0.0,0.0
3,2017-01-23 00:00:00+00:00,1.000000,0.493900,0.0,0.160000,0.840000,0.000000,1,0,1,0,155135,22161,177296,24.000000,0,0,2,0,0,1,0,0,17.907206,17.907206,12.085582,12.085582,0.0,0.0,0.000000,1.000000,0.000000,1.000000,0.0,0.0
4,2017-01-24 00:00:00+00:00,0.750000,0.243825,0.0,0.122750,0.849250,0.028000,4,0,3,1,431555,71637,503192,21.000000,1,1,1,0,0,1,0,0,3.655768,14.623071,2.987519,11.950077,0.0,0.0,0.000000,0.750000,0.250000,0.250000,0.0,0.0


## 14) Prepare Daily Oil Data
Critical fix: the target is **next-day oil return**, not same-day return using same-day high/low/open/close as predictors.

In [ ]:
def normalize_colnames(data):
    data = data.copy()
    data.columns = [str(c).strip() for c in data.columns]
    return data

def find_col(columns, candidates):
    lower_map = {str(c).lower(): c for c in columns}
    for cand in candidates:
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    for c in columns:
        cl = str(c).lower()
        if any(cand.lower() in cl for cand in candidates):
            return c
    return None

oil_daily = normalize_colnames(oil_daily_raw)

date_col = find_col(oil_daily.columns, ['Date', 'Datetime', 'date', 'datetime'])
open_col = find_col(oil_daily.columns, ['Open', 'open'])
high_col = find_col(oil_daily.columns, ['High', 'high'])
low_col = find_col(oil_daily.columns, ['Low', 'low'])
close_col = find_col(oil_daily.columns, ['Close', 'Adj Close', 'close'])
volume_col = find_col(oil_daily.columns, ['Volume', 'volume'])

needed = {
    'date_col': date_col,
    'open_col': open_col,
    'high_col': high_col,
    'low_col': low_col,
    'close_col': close_col,
}
missing = [k for k, v in needed.items() if v is None]
if missing:
    raise ValueError(f'Could not detect required oil columns: {missing}. Available columns: {oil_daily.columns.tolist()}')

oil_daily = oil_daily.rename(columns={
    date_col: 'Date',
    open_col: 'oil_open',
    high_col: 'oil_high',
    low_col: 'oil_low',
    close_col: 'oil_close',
})

if volume_col is not None:
    oil_daily = oil_daily.rename(columns={volume_col: 'oil_volume'})
else:
    oil_daily['oil_volume'] = 0

oil_daily['Date'] = pd.to_datetime(oil_daily['Date'], errors='coerce', utc=True)
oil_daily = oil_daily.dropna(subset=['Date']).copy()
oil_daily['day'] = oil_daily['Date'].dt.floor('D')

for col in ['oil_open', 'oil_high', 'oil_low', 'oil_close', 'oil_volume']:
    oil_daily[col] = pd.to_numeric(oil_daily[col], errors='coerce')

oil_daily = oil_daily.dropna(subset=['oil_open', 'oil_high', 'oil_low', 'oil_close']).copy()
oil_daily = oil_daily.sort_values('day').drop_duplicates(subset=['day'], keep='last')

oil_daily['oil_return_pct_same_day'] = ((oil_daily['oil_close'] - oil_daily['oil_open']) / oil_daily['oil_open']) * 100
oil_daily['oil_delta_same_day'] = oil_daily['oil_close'] - oil_daily['oil_open']
oil_daily['oil_volatility_same_day'] = oil_daily['oil_high'] - oil_daily['oil_low']

for lag in [1, 2, 3, 5, 7, 10, 14]:
    oil_daily[f'oil_return_lag{lag}'] = oil_daily['oil_return_pct_same_day'].shift(lag)
    oil_daily[f'oil_close_lag{lag}'] = oil_daily['oil_close'].shift(lag)
    oil_daily[f'oil_volume_lag{lag}'] = oil_daily['oil_volume'].shift(lag)
    oil_daily[f'oil_volatility_lag{lag}'] = oil_daily['oil_volatility_same_day'].shift(lag)

for window in [3, 5, 7, 14]:
    oil_daily[f'oil_return_roll_mean_{window}'] = oil_daily['oil_return_pct_same_day'].shift(1).rolling(window, min_periods=1).mean()
    oil_daily[f'oil_return_roll_std_{window}'] = oil_daily['oil_return_pct_same_day'].shift(1).rolling(window, min_periods=2).std()
    oil_daily[f'oil_volatility_roll_mean_{window}'] = oil_daily['oil_volatility_same_day'].shift(1).rolling(window, min_periods=1).mean()

# Target: next available trading day movement.
oil_daily['target_next_day_return_pct'] = oil_daily['oil_return_pct_same_day'].shift(-1)
oil_daily['target_next_day_direction'] = (oil_daily['target_next_day_return_pct'] > 0).astype(int)
oil_daily['target_next_day'] = oil_daily['day'].shift(-1)

safe_oil_cols = ['day', 'target_next_day', 'target_next_day_return_pct', 'target_next_day_direction'] + [
    c for c in oil_daily.columns
    if c.startswith('oil_return_lag')
    or c.startswith('oil_close_lag')
    or c.startswith('oil_volume_lag')
    or c.startswith('oil_volatility_lag')
    or c.startswith('oil_return_roll')
    or c.startswith('oil_volatility_roll')
]

oil_model_daily = oil_daily[safe_oil_cols].copy()
print('Daily oil model data:', oil_model_daily.shape)
display(oil_model_daily.head(10))

Daily oil model data: (2252, 44)


,day,target_next_day,target_next_day_return_pct,target_next_day_direction,oil_return_lag1,oil_close_lag1,oil_volume_lag1,oil_volatility_lag1,oil_return_lag2,oil_close_lag2,oil_volume_lag2,oil_volatility_lag2,oil_return_lag3,oil_close_lag3,oil_volume_lag3,oil_volatility_lag3,oil_return_lag5,oil_close_lag5,oil_volume_lag5,oil_volatility_lag5,oil_return_lag7,oil_close_lag7,oil_volume_lag7,oil_volatility_lag7,oil_return_lag10,oil_close_lag10,oil_volume_lag10,oil_volatility_lag10,oil_return_lag14,oil_close_lag14,oil_volume_lag14,oil_volatility_lag14,oil_return_roll_mean_3,oil_return_roll_std_3,oil_volatility_roll_mean_3,oil_return_roll_mean_5,oil_return_roll_std_5,oil_volatility_roll_mean_5,oil_return_roll_mean_7,oil_return_roll_std_7,oil_volatility_roll_mean_7,oil_return_roll_mean_14,oil_return_roll_std_14,oil_volatility_roll_mean_14
0,2017-01-20 00:00:00+00:00,2017-01-23 00:00:00+00:00,-1.087571,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2017-01-23 00:00:00+00:00,2017-01-24 00:00:00+00:00,0.605372,1,1.885320,52.419998,567231.0,1.510002,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.885320,NaN,1.510002,1.885320,NaN,1.510002,1.885320,NaN,1.510002,1.885320,NaN,1.510002
2,2017-01-24 00:00:00+00:00,2017-01-25 00:00:00+00:00,-0.377716,0,-1.087571,52.750000,455333.0,1.260002,1.885320,52.419998,567231.0,1.510002,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.398875,2.102152,1.385002,0.398875,2.102152,1.385002,0.398875,2.102152,1.385002,0.398875,2.102152,1.385002
3,2017-01-25 00:00:00+00:00,2017-01-26 00:00:00+00:00,1.548338,1,0.605372,53.180000,520285.0,0.890003,-1.087571,52.750000,455333.0,1.260002,1.885320,52.419998,567231.0,1.510002,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.467707,1.491219,1.220002,0.467707,1.491219,1.220002,0.467707,1.491219,1.220002,0.467707,1.491219,1.220002
4,2017-01-26 00:00:00+00:00,2017-01-27 00:00:00+00:00,-1.060669,0,-0.377716,52.750000,589709.0,0.910000,0.605372,53.180000,520285.0,0.890003,-1.087571,52.750000,455333.0,1.260002,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.286639,0.850139,1.020002,0.256351,1.288866,1.142502,0.256351,1.288866,1.142502,0.256351,1.288866,1.142502
5,2017-01-27 00:00:00+00:00,2017-01-30 00:00:00+00:00,-0.978364,0,1.548338,53.779999,578065.0,1.270000,-0.377716,52.750000,589709.0,0.910000,0.605372,53.180000,520285.0,0.890003,1.885320,52.419998,567231.0,1.510002,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.591998,0.963097,1.023335,0.514749,1.256872,1.168002,0.514749,1.256872,1.168002,0.514749,1.256872,1.168002
6,2017-01-30 00:00:00+00:00,2017-01-31 00:00:00+00:00,0.399245,1,-1.060669,53.169998,495556.0,1.500000,1.548338,53.779999,578065.0,1.270000,-0.377716,52.750000,589709.0,0.910000,-1.087571,52.750000,455333.0,1.260002,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.036651,1.352961,1.226667,-0.074449,1.138707,1.166001,0.252179,1.295160,1.223335,0.252179,1.295160,1.223335
7,2017-01-31 00:00:00+00:00,2017-02-01 00:00:00+00:00,2.122826,1,-0.978364,52.630001,486309.0,1.049999,-1.060669,53.169998,495556.0,1.500000,1.548338,53.779999,578065.0,1.270000,0.605372,53.180000,520285.0,0.890003,1.885320,52.419998,567231.0,1.510002,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.163565,1.483122,1.273333,-0.052608,1.115221,1.124001,0.076387,1.270506,1.198572,0.076387,1.270506,1.198572
8,2017-02-01 00:00:00+00:00,2017-02-02 00:00:00+00:00,-0.055999,0,0.399245,52.810001,586676.0,1.320000,-0.978364,52.630001,486309.0,1.049999,-1.060669,53.169998,495556.0,1.500000,-0.377716,52.750000,589709.0,0.910000,-1.087571,52.750000,455333.0,1.260002,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.546596,0.820155,1.290000,-0.093833,1.088302,1.210000,-0.135909,1.016663,1.171429,0.116744,1.181785,1.213751
9,2017-02-02 00:00:00+00:00,2017-02-03 00:00:00+00:00,0.279437,1,2.122826,53.880

## 15) Merge Daily Tweets with Safe Future Oil Target
For day `t`, the model uses:
- tweet features from day `t`
- oil lag features from days before `t`

to predict oil movement on the next available trading day.

In [ ]:
daily_merged = pd.merge(daily_tweets, oil_model_daily, on='day', how='inner')
daily_merged = daily_merged.sort_values('day').dropna(subset=['target_next_day_return_pct']).copy()

# Tweet lag features.
tweet_lag_base_cols = [
    'avg_sentiment', 'avg_polarity', 'tweet_count', 'negative_tweets', 'positive_tweets',
    'market_keyword_mentions', 'geo_keyword_mentions', 'oil_direct_keyword_mentions',
    'market_related_posts', 'geo_related_posts', 'oil_direct_related_posts',
    'total_engagement', 'total_final_impact', 'total_market_importance', 'total_oil_direct_importance'
]

tweet_lag_base_cols = [c for c in tweet_lag_base_cols if c in daily_merged.columns]

for lag in [1, 2, 3, 5, 7, 14]:
    for col in tweet_lag_base_cols:
        daily_merged[f'{col}_lag{lag}'] = daily_merged[col].shift(lag)

# Rolling features based only on previous rows.
rolling_base_cols = [
    'avg_sentiment', 'avg_polarity', 'tweet_count', 'market_keyword_mentions',
    'geo_keyword_mentions', 'oil_direct_keyword_mentions', 'total_engagement',
    'total_final_impact', 'total_market_importance', 'total_oil_direct_importance'
]
rolling_base_cols = [c for c in rolling_base_cols if c in daily_merged.columns]

for window in [3, 7, 14]:
    for col in rolling_base_cols:
        daily_merged[f'{col}_roll_mean_{window}'] = daily_merged[col].shift(1).rolling(window=window, min_periods=1).mean()
        daily_merged[f'{col}_roll_sum_{window}'] = daily_merged[col].shift(1).rolling(window=window, min_periods=1).sum()

# Drop rows created by lags/rolling oil std.
daily_merged = daily_merged.replace([np.inf, -np.inf], np.nan).dropna().copy()

print('Merged daily modeling rows:', daily_merged.shape)
print('Date range:', daily_merged['day'].min(), 'to', daily_merged['day'].max())
display(daily_merged.head())

Merged daily modeling rows: (1903, 228)
Date range: 2017-02-09 00:00:00+00:00 to 2025-12-30 00:00:00+00:00


,day,avg_sentiment,avg_polarity,avg_subjectivity,avg_positive_score,avg_neutral_score,avg_negative_score,tweet_count,negative_tweets,positive_tweets,neutral_tweets,total_likes,total_reposts,total_engagement,avg_word_count,hashtag_count,mention_count,market_keyword_mentions,geo_keyword_mentions,oil_direct_keyword_mentions,market_related_posts,geo_related_posts,oil_direct_related_posts,avg_final_impact,total_final_impact,avg_market_importance,total_market_importance,avg_oil_direct_importance,total_oil_direct_importance,negative_ratio,positive_ratio,neutral_ratio,market_related_ratio,geo_related_ratio,oil_direct_related_ratio,target_next_day,target_next_day_return_pct,target_next_day_direction,oil_return_lag1,oil_close_lag1,oil_volume_lag1,oil_volatility_lag1,oil_return_lag2,oil_close_lag2,oil_volume_lag2,oil_volatility_lag2,oil_return_lag3,oil_close_lag3,oil_volume_lag3,oil_volatility_lag3,oil_return_lag5,oil_close_lag5,oil_volume_lag5,oil_volatility_lag5,oil_return_lag7,oil_close_lag7,oil_volume_lag7,oil_volatility_lag7,oil_return_lag10,oil_close_lag10,oil_volume_lag10,oil_volatility_lag10,oil_return_lag14,oil_close_lag14,oil_volume_lag14,oil_volatility_lag14,oil_return_roll_mean_3,oil_return_roll_std_3,oil_volatility_roll_mean_3,oil_return_roll_mean_5,oil_return_roll_std_5,oil_volatility_roll_mean_5,oil_return_roll_mean_7,oil_return_roll_std_7,oil_volatility_roll_mean_7,oil_return_roll_mean_14,oil_return_roll_std_14,oil_volatility_roll_mean_14,avg_sentiment_lag1,avg_polarity_lag1,...,oil_direct_related_posts_lag7,total_engagement_lag7,total_final_impact_lag7,total_market_importance_lag7,total_oil_direct_importance_lag7,avg_sentiment_lag14,avg_polarity_lag14,tweet_count_lag14,negative_tweets_lag14,positive_tweets_lag14,market_keyword_mentions_lag14,geo_keyword_mentions_lag14,oil_direct_keyword_mentions_lag14,market_related_posts_lag14,geo_related_posts_lag14,oil_direct_related_posts_lag14,total_engagement_lag14,total_final_impact_lag14,total_market_importance_lag14,total_oil_direct_importance_lag14,avg_sentiment_roll_mean_3,avg_sentiment_roll_sum_3,avg_polarity_roll_mean_3,avg_polarity_roll_sum_3,tweet_count_roll_mean_3,tweet_count_roll_sum_3,market_keyword_mentions_roll_mean_3,market_keyword_mentions_roll_sum_3,geo_keyword_mentions_roll_mean_3,geo_keyword_mentions_roll_sum_3,oil_direct_keyword_mentions_roll_mean_3,oil_direct_keyword_mentions_roll_sum_3,total_engagement_roll_mean_3,total_engagement_roll_sum_3,total_final_impact_roll_mean_3,total_final_impact_roll_sum_3,total_market_importance_roll_mean_3,total_market_importance_roll_sum_3,total_oil_direct_importance_roll_mean_3,total_oil_direct_importance_roll_sum_3,avg_sentiment_roll_mean_7,avg_sentiment_roll_sum_7,avg_polarity_roll_mean_7,avg_polarity_roll_sum_7,tweet_count_roll_mean_7,tweet_count_roll_sum_7,market_keyword_mentions_roll_mean_7,market_keyword_mentions_roll_sum_7,geo_keyword_mentions_roll_mean_7,geo_keyword_mentions_roll_sum_7,oil_direct_keyword_mentions_roll_mean_7,oil_direct_keyword_mentions_roll_sum_7,total_engagement_roll_mean_7,total_engagement_roll_sum_7,total_final_impact_roll_mean_7,total_final_impact_roll_sum_7,total_market_importance_roll_mean_7,total_market_importance_roll_sum_7,total_oil_direct_importance_roll_mean_7,total_oil_direct_importance_roll_sum_7,avg_sentiment_roll_mean_14,avg_sentiment_roll_sum_14,avg_polarity_roll_mean_14,avg_polarity_roll_sum_14,tweet_count_roll_mean_14,tweet_count_roll_sum_14,market_keyword_mentions_roll_mean_14,market_keyword_mentions_roll_sum_14,geo_keyword_mentions_roll_mean_14,geo_keyword_mentions_roll_sum_14,oil_direct_keyword_mentions_roll_mean_14,oil_direct_keyword_mentions_roll_sum_14,total_engagement_roll_mean_14,total_engagement_roll_sum_14,total_final_impact_roll_mean_14,total_final_impact_roll_sum_14,total_market_importance_roll_mean_14,total_market_importance_roll_sum_14,total_oil_direct_importance_roll_mean_14,total_oil_direct_importance_roll_sum_14
14,2017-02-09 00:00:00+00:00,0.750000,0.249813,0.0,0.1975

## 15.1) Added Improvement — Sentiment Shock Analysis
This cell checks whether extreme positive or negative tweet-sentiment days are linked with different next-day oil returns and volatility-related signals.


In [ ]:
# =========================
# Added Improvement: Sentiment Shock Features + Oil Volatility Link
# Place: directly after the daily_merged creation cell
# =========================

shock_df = daily_merged.copy()

# Define sentiment shock thresholds using quantiles.
neg_threshold = shock_df['avg_sentiment'].quantile(0.10)
pos_threshold = shock_df['avg_sentiment'].quantile(0.90)

shock_df['negative_sentiment_shock'] = shock_df['avg_sentiment'] <= neg_threshold
shock_df['positive_sentiment_shock'] = shock_df['avg_sentiment'] >= pos_threshold

# Detect volatility columns already created in the notebook.
volatility_cols = [c for c in shock_df.columns if 'oil_volatility' in c.lower()]

print('Negative sentiment threshold:', round(neg_threshold, 4))
print('Positive sentiment threshold:', round(pos_threshold, 4))
print('Available volatility columns:')
print(volatility_cols)

agg_dict = {
    'days': ('day', 'count'),
    'avg_next_day_return': ('target_next_day_return_pct', 'mean'),
    'median_next_day_return': ('target_next_day_return_pct', 'median'),
    'avg_tweet_count': ('tweet_count', 'mean'),
    'avg_negative_tweets': ('negative_tweets', 'mean'),
    'avg_positive_tweets': ('positive_tweets', 'mean'),
    'avg_market_mentions': ('market_keyword_mentions', 'mean'),
    'avg_oil_mentions': ('oil_direct_keyword_mentions', 'mean')
}

# Add the first available volatility column to the summary if it exists.
if volatility_cols:
    agg_dict['avg_oil_volatility_proxy'] = (volatility_cols[0], 'mean')

shock_summary = shock_df.groupby(
    ['negative_sentiment_shock', 'positive_sentiment_shock']
).agg(**agg_dict).reset_index()

display(shock_summary)


## 16) Leakage Audit
This blocks the most dangerous mistake: using current-day/future oil columns as predictors.

In [ ]:
leakage_forbidden_features = [
    'oil_open', 'oil_high', 'oil_low', 'oil_close', 'oil_volume',
    'oil_return_pct_same_day', 'oil_delta_same_day', 'oil_volatility_same_day',
    'target_next_day_return_pct', 'target_next_day_direction', 'target_next_day'
]

print('Forbidden raw/current/future oil columns that must NOT be in X:')
print(leakage_forbidden_features)

Forbidden raw/current/future oil columns that must NOT be in X:
['oil_open', 'oil_high', 'oil_low', 'oil_close', 'oil_volume', 'oil_return_pct_same_day', 'oil_delta_same_day', 'oil_volatility_same_day', 'target_next_day_return_pct', 'target_next_day_direction', 'target_next_day']


## 17) Chronological Train/Test Split
Time series data must be split by time. Older dates train the model; newer dates test it.

In [ ]:
def chronological_train_test_split(data, date_col='day', test_size=0.2):
    data = data.sort_values(date_col).copy()
    split_idx = int(len(data) * (1 - test_size))
    train = data.iloc[:split_idx].copy()
    test = data.iloc[split_idx:].copy()
    if len(train) == 0 or len(test) == 0:
        raise ValueError('Train or test split is empty. Check dataset size or test_size.')
    if train[date_col].max() >= test[date_col].min():
        raise ValueError('Chronological split failed: train period overlaps test period.')
    return train, test

train_df, test_df = chronological_train_test_split(daily_merged, date_col='day', test_size=TEST_SIZE)

print('Train rows:', train_df.shape[0])
print('Test rows:', test_df.shape[0])
print('Train period:', train_df['day'].min(), 'to', train_df['day'].max())
print('Test period:', test_df['day'].min(), 'to', test_df['day'].max())
print('Chronological split check:', train_df['day'].max() < test_df['day'].min())

Train rows: 1522
Test rows: 381
Train period: 2017-02-09 00:00:00+00:00 to 2024-06-24 00:00:00+00:00
Test period: 2024-06-25 00:00:00+00:00 to 2025-12-30 00:00:00+00:00
Chronological split check: True


## 18) Distribution Shift Check
This matters because Trump moved from Twitter to Truth Social. If the platform distribution changes, model performance can drop even if the code is correct.

In [ ]:
platform_daily = df.groupby(['date_day', 'platform']).size().reset_index(name='posts')
platform_pivot = platform_daily.pivot_table(index='date_day', columns='platform', values='posts', fill_value=0).reset_index()
platform_pivot = platform_pivot.rename(columns={'date_day': 'day'})

platform_cols = [c for c in platform_pivot.columns if c != 'day']
platform_pivot['dominant_platform'] = platform_pivot[platform_cols].idxmax(axis=1) if platform_cols else 'Unknown'

train_platform = platform_pivot[platform_pivot['day'].isin(train_df['day'])]['dominant_platform'].value_counts(normalize=True).rename('Train Share')
test_platform = platform_pivot[platform_pivot['day'].isin(test_df['day'])]['dominant_platform'].value_counts(normalize=True).rename('Test Share')
platform_shift = pd.concat([train_platform, test_platform], axis=1).fillna(0)

display(platform_shift)

,Train Share,Test Share
dominant_platform,,
Twitter,0.644547,0.002625
Truth Social,0.355453,0.997375


## 19) Define Feature Sets
We build three feature groups:

1. **Dummy**: no real features, just a benchmark.
2. **Oil-only baseline**: previous oil behavior only.
3. **Tweet/Sentiment model**: oil lags + tweet/sentiment/market features.

The question is not “Can we get any accuracy?” The question is: **do tweet features beat the oil-only baseline?**

In [ ]:
baseline_features = [
    c for c in daily_merged.columns
    if c.startswith('oil_return_lag')
    or c.startswith('oil_close_lag')
    or c.startswith('oil_volume_lag')
    or c.startswith('oil_volatility_lag')
    or c.startswith('oil_return_roll')
    or c.startswith('oil_volatility_roll')
]

same_day_tweet_features = [
    'avg_sentiment', 'avg_polarity', 'avg_subjectivity', 'avg_positive_score', 'avg_neutral_score', 'avg_negative_score',
    'tweet_count', 'negative_tweets', 'positive_tweets', 'neutral_tweets',
    'negative_ratio', 'positive_ratio', 'neutral_ratio',
    'total_likes', 'total_reposts', 'total_engagement',
    'avg_word_count', 'hashtag_count', 'mention_count',
    'market_keyword_mentions', 'geo_keyword_mentions', 'oil_direct_keyword_mentions',
    'market_related_posts', 'geo_related_posts', 'oil_direct_related_posts',
    'market_related_ratio', 'geo_related_ratio', 'oil_direct_related_ratio',
    'avg_final_impact', 'total_final_impact',
    'avg_market_importance', 'total_market_importance',
    'avg_oil_direct_importance', 'total_oil_direct_importance'
]
same_day_tweet_features = [c for c in same_day_tweet_features if c in daily_merged.columns]

lagged_tweet_features = [
    c for c in daily_merged.columns
    if any(c.endswith(f'_lag{lag}') for lag in [1, 2, 3, 5, 7, 14])
    or any(c.endswith(f'_roll_mean_{window}') for window in [3, 7, 14])
    or any(c.endswith(f'_roll_sum_{window}') for window in [3, 7, 14])
]

sentiment_features = baseline_features + same_day_tweet_features + lagged_tweet_features

# Remove forbidden columns in case they accidentally entered.
baseline_features = [c for c in baseline_features if c not in leakage_forbidden_features]
sentiment_features = [c for c in sentiment_features if c not in leakage_forbidden_features]

bad_base = sorted(set(baseline_features).intersection(leakage_forbidden_features))
bad_sent = sorted(set(sentiment_features).intersection(leakage_forbidden_features))
assert not bad_base, f'Leakage in baseline features: {bad_base}'
assert not bad_sent, f'Leakage in sentiment features: {bad_sent}'

print('Baseline feature count:', len(baseline_features))
print('Sentiment feature count:', len(sentiment_features))
print('\nBaseline features:')
print(baseline_features)
print('\nFirst 40 sentiment features:')
print(sentiment_features[:40])

Baseline feature count: 40
Sentiment feature count: 254

Baseline features:
['oil_return_lag1', 'oil_close_lag1', 'oil_volume_lag1', 'oil_volatility_lag1', 'oil_return_lag2', 'oil_close_lag2', 'oil_volume_lag2', 'oil_volatility_lag2', 'oil_return_lag3', 'oil_close_lag3', 'oil_volume_lag3', 'oil_volatility_lag3', 'oil_return_lag5', 'oil_close_lag5', 'oil_volume_lag5', 'oil_volatility_lag5', 'oil_return_lag7', 'oil_close_lag7', 'oil_volume_lag7', 'oil_volatility_lag7', 'oil_return_lag10', 'oil_close_lag10', 'oil_volume_lag10', 'oil_volatility_lag10', 'oil_return_lag14', 'oil_close_lag14', 'oil_volume_lag14', 'oil_volatility_lag14', 'oil_return_roll_mean_3', 'oil_return_roll_std_3', 'oil_volatility_roll_mean_3', 'oil_return_roll_mean_5', 'oil_return_roll_std_5', 'oil_volatility_roll_mean_5', 'oil_return_roll_mean_7', 'oil_return_roll_std_7', 'oil_volatility_roll_mean_7', 'oil_return_roll_mean_14', 'oil_return_roll_std_14', 'oil_volatility_roll_mean_14']

First 40 sentiment features:
['oil

## 20) Prepare X and y Safely
The scaler is fitted only on training data, then applied to test data.

In [ ]:
from sklearn.preprocessing import StandardScaler

Xb_train = train_df[baseline_features].copy()
Xb_test = test_df[baseline_features].copy()
Xs_train = train_df[sentiment_features].copy()
Xs_test = test_df[sentiment_features].copy()

y_reg_train = train_df['target_next_day_return_pct'].copy()
y_reg_test = test_df['target_next_day_return_pct'].copy()

y_clf_train = train_df['target_next_day_direction'].copy()
y_clf_test = test_df['target_next_day_direction'].copy()

for data_part in [Xb_train, Xb_test, Xs_train, Xs_test]:
    data_part.replace([np.inf, -np.inf], np.nan, inplace=True)
    data_part.fillna(0, inplace=True)

scaler_base = StandardScaler()
scaler_sent = StandardScaler()

Xb_train_scaled = scaler_base.fit_transform(Xb_train)
Xb_test_scaled = scaler_base.transform(Xb_test)

Xs_train_scaled = scaler_sent.fit_transform(Xs_train)
Xs_test_scaled = scaler_sent.transform(Xs_test)

print('Scaling done using training data only.')
print('Train class balance:')
print(y_clf_train.value_counts(normalize=True).rename('share'))
print('\nTest class balance:')
print(y_clf_test.value_counts(normalize=True).rename('share'))

Scaling done using training data only.
Train class balance:
target_next_day_direction
1    0.521682
0    0.478318
Name: share, dtype: float64

Test class balance:
target_next_day_direction
0    0.535433
1    0.464567
Name: share, dtype: float64


## 21) Regression Models — Predict Next-Day Oil Return %
This answers: can the features estimate the size of next-day oil return?

Expected reality: oil returns are noisy, so regression may be weak. Negative R² is possible and should not be hidden.

In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge, HuberRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

regression_models = {
    'Dummy Mean': ('baseline', DummyRegressor(strategy='mean')),
    'Oil Baseline Ridge': ('baseline', Ridge(alpha=1.0, random_state=RANDOM_STATE)),
    'Oil Baseline RF': ('baseline', RandomForestRegressor(n_estimators=250, max_depth=5, min_samples_leaf=5, random_state=RANDOM_STATE, n_jobs=-1)),
    'Sentiment Ridge': ('sentiment', Ridge(alpha=1.0, random_state=RANDOM_STATE)),
    'Sentiment Huber': ('sentiment', HuberRegressor(max_iter=500)),
    'Sentiment RF': ('sentiment', RandomForestRegressor(n_estimators=250, max_depth=5, min_samples_leaf=5, random_state=RANDOM_STATE, n_jobs=-1)),
    'Sentiment GB': ('sentiment', GradientBoostingRegressor(n_estimators=150, max_depth=2, learning_rate=0.05, random_state=RANDOM_STATE)),
}

reg_results = []
reg_predictions = {}

for name, (feature_type, model) in regression_models.items():
    if feature_type == 'baseline':
        model.fit(Xb_train_scaled, y_reg_train)
        pred = model.predict(Xb_test_scaled)
    else:
        model.fit(Xs_train_scaled, y_reg_train)
        pred = model.predict(Xs_test_scaled)

    reg_predictions[name] = pred
    rmse = mean_squared_error(y_reg_test, pred) ** 0.5
    reg_results.append({
        'Model': name,
        'MAE': mean_absolute_error(y_reg_test, pred),
        'RMSE': rmse,
        'R2': r2_score(y_reg_test, pred)
    })

reg_results_df = pd.DataFrame(reg_results).sort_values('R2', ascending=False)
display(reg_results_df)

,Model,MAE,RMSE,R2
0,Dummy Mean,1.479707,1.966642,-0.009424
6,Sentiment GB,1.566985,2.041785,-0.088036
2,Oil Baseline RF,1.526898,2.058637,-0.106071
5,Sentiment RF,1.531265,2.062970,-0.110732
4,Sentiment Huber,2.000402,3.491964,-2.182459
1,Oil Baseline Ridge,2.929712,3.848769,-2.866045
3,Sentiment Ridge,5.255191,10.827748,-29.598465


## 22) Regression Interpretation Guardrail
This cell prevents overclaiming.

In [ ]:
best_reg = reg_results_df.iloc[0]
print('Best regression model:', best_reg['Model'])
print('Best R2:', round(best_reg['R2'], 4))
print('Best MAE:', round(best_reg['MAE'], 4))

if best_reg['R2'] < 0:
    print('\nInterpretation: Regression performance is weak. The model does not explain next-day oil return better than a simple average benchmark.')
elif best_reg['R2'] < 0.1:
    print('\nInterpretation: Regression signal exists but is weak. Do not overclaim predictive power.')
else:
    print('\nInterpretation: Regression has some useful signal, but still needs business caution and external variables.')

if PLOTLY_OK:
    fig = px.bar(reg_results_df, x='Model', y='R2', text='R2', title='Regression Model Comparison — R²')
    fig.update_traces(texttemplate='%{text:.3f}', textposition='outside')
    style_fig(fig, height=520).show()
else:
    reg_results_df.plot(kind='bar', x='Model', y='R2', title='Regression Model Comparison — R²')
    plt.xticks(rotation=45)
    plt.show()

Best regression model: Dummy Mean
Best R2: -0.0094
Best MAE: 1.4797

Interpretation: Regression performance is weak. The model does not explain next-day oil return better than a simple average benchmark.


## 23) Classification Models — Predict Next-Day Up/Down Direction
This answers: can the features predict whether oil goes up or down tomorrow?

Balanced Accuracy is emphasized because raw accuracy can be misleading when classes are imbalanced.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score,
    f1_score, classification_report, confusion_matrix, roc_auc_score
)

classification_models = {
    'Dummy Majority': ('baseline', DummyClassifier(strategy='most_frequent')),
    'Oil Baseline Logistic': ('baseline', LogisticRegression(max_iter=2000, class_weight='balanced', random_state=RANDOM_STATE)),
    'Oil Baseline RF': ('baseline', RandomForestClassifier(n_estimators=250, max_depth=5, min_samples_leaf=5, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1)),
    'Sentiment Logistic': ('sentiment', LogisticRegression(max_iter=2000, class_weight='balanced', random_state=RANDOM_STATE)),
    'Sentiment RF': ('sentiment', RandomForestClassifier(n_estimators=250, max_depth=5, min_samples_leaf=5, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1)),
    'Sentiment GB': ('sentiment', GradientBoostingClassifier(n_estimators=150, max_depth=2, learning_rate=0.05, random_state=RANDOM_STATE)),
}

clf_results = []
clf_predictions = {}
clf_probabilities = {}

for name, (feature_type, model) in classification_models.items():
    if feature_type == 'baseline':
        model.fit(Xb_train_scaled, y_clf_train)
        pred = model.predict(Xb_test_scaled)
        if hasattr(model, 'predict_proba'):
            prob = model.predict_proba(Xb_test_scaled)[:, 1]
        else:
            prob = None
    else:
        model.fit(Xs_train_scaled, y_clf_train)
        pred = model.predict(Xs_test_scaled)
        if hasattr(model, 'predict_proba'):
            prob = model.predict_proba(Xs_test_scaled)[:, 1]
        else:
            prob = None

    clf_predictions[name] = pred
    clf_probabilities[name] = prob

    try:
        auc = roc_auc_score(y_clf_test, prob) if prob is not None else np.nan
    except Exception:
        auc = np.nan

    clf_results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_clf_test, pred),
        'Balanced Accuracy': balanced_accuracy_score(y_clf_test, pred),
        'Precision': precision_score(y_clf_test, pred, zero_division=0),
        'Recall': recall_score(y_clf_test, pred, zero_division=0),
        'F1': f1_score(y_clf_test, pred, zero_division=0),
        'ROC AUC': auc
    })

clf_results_df = pd.DataFrame(clf_results).sort_values('Balanced Accuracy', ascending=False)
display(clf_results_df)

,Model,Accuracy,Balanced Accuracy,Precision,Recall,F1,ROC AUC
4,Sentiment RF,0.519685,0.525673,0.486486,0.610169,0.541353,0.517863
5,Sentiment GB,0.506562,0.518278,0.478261,0.683616,0.562791,0.530686
2,Oil Baseline RF,0.496063,0.513709,0.473684,0.762712,0.584416,0.511743
3,Sentiment Logistic,0.501312,0.510386,0.472803,0.638418,0.543269,0.497840
1,Oil Baseline Logistic,0.480315,0.501620,0.465574,0.802260,0.589212,0.535089
0,Dummy Majority,0.464567,0.500000,0.464567,1.000000,0.634409,0.500000


## 24) Classification Report for Best Model
This shows exactly where the model fails: false positives, false negatives, and class imbalance.

In [ ]:
best_clf_name = clf_results_df.iloc[0]['Model']
best_clf_pred = clf_predictions[best_clf_name]

print('Best classification model:', best_clf_name)
print('\nClassification report:')
print(classification_report(y_clf_test, best_clf_pred, target_names=['Down/Flat', 'Up'], zero_division=0))

cm = confusion_matrix(y_clf_test, best_clf_pred)
cm_df = pd.DataFrame(cm, index=['Actual Down/Flat', 'Actual Up'], columns=['Pred Down/Flat', 'Pred Up'])
display(cm_df)

if PLOTLY_OK:
    fig = px.imshow(cm_df, text_auto=True, title=f'Confusion Matrix — {best_clf_name}')
    style_fig(fig, height=500).show()

Best classification model: Sentiment RF

Classification report:
              precision    recall  f1-score   support

   Down/Flat       0.57      0.44      0.50       204
          Up       0.49      0.61      0.54       177

    accuracy                           0.52       381
   macro avg       0.53      0.53      0.52       381
weighted avg       0.53      0.52      0.52       381



,Pred Down/Flat,Pred Up
Actual Down/Flat,90,114
Actual Up,69,108


## 24.1) Added Improvement — Stronger Quantitative Model Evaluation
This cell expands the classification evaluation beyond accuracy, which is important because financial direction prediction can be noisy and class balance can be imperfect.


In [ ]:
# =========================
# Added Improvement: Stronger Quantitative Model Evaluation
# Place: directly after the best classification model report
# =========================

strong_eval_rows = []

for model_name, preds in clf_predictions.items():
    probs = clf_probabilities.get(model_name)

    row = {
        'Model': model_name,
        'Accuracy': accuracy_score(y_clf_test, preds),
        'Balanced Accuracy': balanced_accuracy_score(y_clf_test, preds),
        'Precision': precision_score(y_clf_test, preds, zero_division=0),
        'Recall': recall_score(y_clf_test, preds, zero_division=0),
        'F1 Score': f1_score(y_clf_test, preds, zero_division=0)
    }

    if probs is not None:
        try:
            row['ROC AUC'] = roc_auc_score(y_clf_test, probs)
        except Exception:
            row['ROC AUC'] = np.nan
    else:
        row['ROC AUC'] = np.nan

    strong_eval_rows.append(row)

strong_eval_df = pd.DataFrame(strong_eval_rows).sort_values(
    by=['Balanced Accuracy', 'F1 Score'],
    ascending=False
)

display(strong_eval_df)

if PLOTLY_OK:
    metric_viz = strong_eval_df.melt(
        id_vars='Model',
        value_vars=['Accuracy', 'Balanced Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC AUC'],
        var_name='Metric',
        value_name='Score'
    )

    fig = px.bar(
        metric_viz,
        x='Model',
        y='Score',
        color='Metric',
        barmode='group',
        title='Full Classification Model Evaluation'
    )

    fig.update_layout(xaxis_tickangle=-35, height=600)
    style_fig(fig, height=600).show()


## 25) Honest Baseline Comparison
This directly checks whether tweet/sentiment features add value beyond the oil-only baseline.

In [ ]:
baseline_only = clf_results_df[clf_results_df['Model'].str.startswith('Oil Baseline')].sort_values('Balanced Accuracy', ascending=False)
sentiment_only = clf_results_df[clf_results_df['Model'].str.startswith('Sentiment')].sort_values('Balanced Accuracy', ascending=False)
dummy_only = clf_results_df[clf_results_df['Model'].str.startswith('Dummy')].head(1)

comparison_summary = pd.concat([dummy_only, baseline_only.head(1), sentiment_only.head(1)], ignore_index=True)
display(comparison_summary)

if not baseline_only.empty and not sentiment_only.empty:
    base_score = baseline_only.iloc[0]['Balanced Accuracy']
    sent_score = sentiment_only.iloc[0]['Balanced Accuracy']
    diff = sent_score - base_score
    print(f'Sentiment improvement over oil-only baseline Balanced Accuracy: {diff:.4f}')

    if diff > 0.03:
        print('Interpretation: Tweet/sentiment features add a meaningful predictive improvement over the oil-only baseline.')
    elif diff > 0.01:
        print('Interpretation: Tweet/sentiment features add a small but visible improvement. Still avoid strong claims.')
    elif diff > 0:
        print('Interpretation: Tweet/sentiment features improve the baseline slightly, but the gain is very small.')
    else:
        print('Interpretation: Tweet/sentiment features do not beat the oil-only baseline. Do not overclaim.')

,Model,Accuracy,Balanced Accuracy,Precision,Recall,F1,ROC AUC
0,Dummy Majority,0.464567,0.500000,0.464567,1.000000,0.634409,0.500000
1,Oil Baseline RF,0.496063,0.513709,0.473684,0.762712,0.584416,0.511743
2,Sentiment RF,0.519685,0.525673,0.486486,0.610169,0.541353,0.517863


Sentiment improvement over oil-only baseline Balanced Accuracy: 0.0120
Interpretation: Tweet/sentiment features add a small but visible improvement. Still avoid strong claims.


## 26) Model Comparison Dashboard

In [ ]:
if PLOTLY_OK:
    metric_cols = ['Accuracy', 'Balanced Accuracy', 'F1', 'ROC AUC']
    melted = clf_results_df.melt(id_vars='Model', value_vars=metric_cols, var_name='Metric', value_name='Score')
    fig = px.bar(melted, x='Model', y='Score', color='Metric', barmode='group', title='Classification Model Comparison')
    style_fig(fig, height=600).show()
else:
    clf_results_df.plot(kind='bar', x='Model', y=['Accuracy', 'Balanced Accuracy', 'F1'], title='Classification Model Comparison')
    plt.xticks(rotation=45)
    plt.show()

## 27) Correlation Analysis — Correctly Framed
Correlations are descriptive. They do not prove causality.

In [ ]:
corr_cols = [
    'target_next_day_return_pct',
    'avg_sentiment', 'avg_polarity', 'tweet_count', 'negative_tweets', 'positive_tweets',
    'market_keyword_mentions', 'geo_keyword_mentions', 'oil_direct_keyword_mentions',
    'market_related_posts', 'geo_related_posts', 'oil_direct_related_posts',
    'total_engagement', 'avg_final_impact', 'total_final_impact',
    'total_market_importance', 'total_oil_direct_importance'
]
corr_cols = [c for c in corr_cols if c in daily_merged.columns]

corr_df = daily_merged[corr_cols].corr(numeric_only=True)[['target_next_day_return_pct']].sort_values('target_next_day_return_pct', ascending=False)
display(corr_df)

if PLOTLY_OK:
    plot_corr = corr_df.drop(index='target_next_day_return_pct', errors='ignore').reset_index()
    plot_corr.columns = ['Feature', 'Correlation']
    fig = px.bar(plot_corr, x='Correlation', y='Feature', orientation='h', title='Correlation with Next-Day Oil Return')
    style_fig(fig, height=650).show()

,target_next_day_return_pct
target_next_day_return_pct,1.000000
avg_final_impact,0.019080
geo_related_posts,0.017808
total_final_impact,0.015836
tweet_count,0.015497
geo_keyword_mentions,0.013139
negative_tweets,0.012242
positive_tweets,0.011550
total_market_importance,0.011065
avg_polarity,0.010800


## 27.1) Added Improvement — Event Study Around Sentiment Shocks
This cell adds a stronger analytical layer by checking average next-day oil returns around extreme positive and negative sentiment days. This supports cautious association analysis, not direct causality claims.


In [ ]:
# =========================
# Added Improvement: Event Study Around Strong Sentiment Shock Days
# Place: directly after the correlation analysis cell
# =========================

event_df = daily_merged.copy().sort_values('day').reset_index(drop=True)

# Define strong sentiment events using the 10th and 90th percentiles.
negative_event_threshold = event_df['avg_sentiment'].quantile(0.10)
positive_event_threshold = event_df['avg_sentiment'].quantile(0.90)

negative_event_days = event_df.loc[
    event_df['avg_sentiment'] <= negative_event_threshold, 'day'
].tolist()

positive_event_days = event_df.loc[
    event_df['avg_sentiment'] >= positive_event_threshold, 'day'
].tolist()

def build_event_window(data, event_days, label, window=3):
    rows = []

    for event_day in event_days:
        event_index = data.index[data['day'] == event_day]

        if len(event_index) == 0:
            continue

        event_index = event_index[0]

        for offset in range(-window, window + 1):
            idx = event_index + offset

            if idx < 0 or idx >= len(data):
                continue

            rows.append({
                'event_day': event_day,
                'relative_day': offset,
                'event_type': label,
                'day': data.loc[idx, 'day'],
                'target_next_day_return_pct': data.loc[idx, 'target_next_day_return_pct'],
                'avg_sentiment': data.loc[idx, 'avg_sentiment'],
                'negative_tweets': data.loc[idx, 'negative_tweets'],
                'positive_tweets': data.loc[idx, 'positive_tweets'],
                'market_keyword_mentions': data.loc[idx, 'market_keyword_mentions'],
                'oil_direct_keyword_mentions': data.loc[idx, 'oil_direct_keyword_mentions']
            })

    return pd.DataFrame(rows)

negative_window = build_event_window(event_df, negative_event_days, 'Negative Sentiment Shock', window=3)
positive_window = build_event_window(event_df, positive_event_days, 'Positive Sentiment Shock', window=3)

event_study_df = pd.concat([negative_window, positive_window], ignore_index=True)

event_summary = event_study_df.groupby(['event_type', 'relative_day']).agg(
    avg_next_day_return=('target_next_day_return_pct', 'mean'),
    median_next_day_return=('target_next_day_return_pct', 'median'),
    avg_sentiment=('avg_sentiment', 'mean'),
    avg_market_mentions=('market_keyword_mentions', 'mean'),
    observations=('target_next_day_return_pct', 'count')
).reset_index()

display(event_summary)

if PLOTLY_OK:
    fig = px.line(
        event_summary,
        x='relative_day',
        y='avg_next_day_return',
        color='event_type',
        markers=True,
        title='Event Study: Average Next-Day Oil Return Around Sentiment Shocks'
    )

    fig.add_vline(x=0, line_dash='dash')
    fig.update_layout(
        xaxis_title='Days Around Sentiment Shock',
        yaxis_title='Average Next-Day Oil Return %',
        height=550
    )

    style_fig(fig, height=550).show()


## 28) Fixed Visualization Scaling
Sentiment and oil return are not on the same scale. This chart uses z-scores for visual comparison only.

In [ ]:
def zscore(s):
    s = pd.Series(s)
    std = s.std()
    if std == 0 or pd.isna(std):
        return s * 0
    return (s - s.mean()) / std

viz_df = daily_merged[['day', 'avg_sentiment', 'target_next_day_return_pct', 'market_keyword_mentions', 'oil_direct_keyword_mentions']].copy()
viz_df['avg_sentiment_z'] = zscore(viz_df['avg_sentiment'])
viz_df['next_day_return_z'] = zscore(viz_df['target_next_day_return_pct'])
viz_df['market_mentions_z'] = zscore(viz_df['market_keyword_mentions'])
viz_df['oil_direct_mentions_z'] = zscore(viz_df['oil_direct_keyword_mentions'])

if PLOTLY_OK:
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=viz_df['day'], y=viz_df['avg_sentiment_z'], mode='lines', name='Avg Sentiment z-score'))
    fig.add_trace(go.Scatter(x=viz_df['day'], y=viz_df['next_day_return_z'], mode='lines', name='Next-Day Oil Return z-score'))
    fig.add_trace(go.Scatter(x=viz_df['day'], y=viz_df['market_mentions_z'], mode='lines', name='Market Keyword Mentions z-score', opacity=0.55))
    style_fig(fig, title='Normalized Sentiment / Market Mentions vs Next-Day Oil Return', height=560).show()
else:
    plt.figure(figsize=(14,5))
    plt.plot(viz_df['day'], viz_df['avg_sentiment_z'], label='Avg Sentiment z-score')
    plt.plot(viz_df['day'], viz_df['next_day_return_z'], label='Next-Day Return z-score')
    plt.plot(viz_df['day'], viz_df['market_mentions_z'], label='Market Mentions z-score', alpha=0.6)
    plt.legend()
    plt.title('Normalized Sentiment / Market Mentions vs Next-Day Oil Return')
    plt.show()

## 28.1) Added Improvement — Clearer Sentiment/Oil Visualizations
These charts make the relationship between negative/positive tweets, market/oil mentions, oil returns, and volatility easier to interpret than a single normalized line chart.


In [ ]:
# =========================
# Added Improvement: Positive/Negative Tweets vs Next-Day Oil Return
# Place: directly after the fixed visualization scaling cell
# =========================

viz_sentiment_oil = daily_merged[
    ['day', 'negative_tweets', 'positive_tweets',
     'negative_ratio', 'positive_ratio',
     'target_next_day_return_pct',
     'market_keyword_mentions', 'oil_direct_keyword_mentions']
].copy()

if PLOTLY_OK:
    fig = go.Figure()

    fig.add_trace(go.Bar(
        x=viz_sentiment_oil['day'],
        y=viz_sentiment_oil['negative_tweets'],
        name='Negative Tweets',
        opacity=0.65
    ))

    fig.add_trace(go.Bar(
        x=viz_sentiment_oil['day'],
        y=viz_sentiment_oil['positive_tweets'],
        name='Positive Tweets',
        opacity=0.65
    ))

    fig.add_trace(go.Scatter(
        x=viz_sentiment_oil['day'],
        y=viz_sentiment_oil['target_next_day_return_pct'],
        name='Next-Day Oil Return %',
        mode='lines',
        yaxis='y2'
    ))

    fig.update_layout(
        title='Negative/Positive Tweets vs Next-Day Oil Return',
        xaxis_title='Date',
        yaxis=dict(title='Tweet Count'),
        yaxis2=dict(
            title='Next-Day Oil Return %',
            overlaying='y',
            side='right'
        ),
        barmode='group',
        height=600
    )

    style_fig(fig, height=600).show()

else:
    plt.figure(figsize=(14, 6))
    plt.bar(viz_sentiment_oil['day'], viz_sentiment_oil['negative_tweets'], label='Negative Tweets', alpha=0.6)
    plt.bar(viz_sentiment_oil['day'], viz_sentiment_oil['positive_tweets'], label='Positive Tweets', alpha=0.6)
    plt.plot(viz_sentiment_oil['day'], viz_sentiment_oil['target_next_day_return_pct'], label='Next-Day Oil Return %')
    plt.legend()
    plt.title('Negative/Positive Tweets vs Next-Day Oil Return')
    plt.show()


In [ ]:
# =========================
# Added Improvement: Market/Oil Mentions vs Oil Volatility Proxy
# Place: directly after the fixed visualization scaling cell
# =========================

vol_cols = [c for c in daily_merged.columns if 'oil_volatility' in c.lower()]
volatility_col = vol_cols[0] if len(vol_cols) > 0 else None

if volatility_col:
    vol_viz = daily_merged[
        ['day', 'market_keyword_mentions', 'oil_direct_keyword_mentions', volatility_col]
    ].copy()

    if PLOTLY_OK:
        fig = go.Figure()

        fig.add_trace(go.Scatter(
            x=vol_viz['day'],
            y=vol_viz['market_keyword_mentions'],
            mode='lines',
            name='Market Keyword Mentions'
        ))

        fig.add_trace(go.Scatter(
            x=vol_viz['day'],
            y=vol_viz['oil_direct_keyword_mentions'],
            mode='lines',
            name='Oil-Direct Keyword Mentions'
        ))

        fig.add_trace(go.Scatter(
            x=vol_viz['day'],
            y=vol_viz[volatility_col],
            mode='lines',
            name=f'{volatility_col}',
            yaxis='y2'
        ))

        fig.update_layout(
            title='Market/Oil Keyword Mentions vs Oil Volatility',
            xaxis_title='Date',
            yaxis=dict(title='Keyword Mentions'),
            yaxis2=dict(
                title='Oil Volatility Proxy',
                overlaying='y',
                side='right'
            ),
            height=600
        )

        style_fig(fig, height=600).show()

    else:
        plt.figure(figsize=(14, 6))
        plt.plot(vol_viz['day'], vol_viz['market_keyword_mentions'], label='Market Keyword Mentions')
        plt.plot(vol_viz['day'], vol_viz['oil_direct_keyword_mentions'], label='Oil-Direct Mentions')
        plt.plot(vol_viz['day'], vol_viz[volatility_col], label=volatility_col)
        plt.legend()
        plt.title('Market/Oil Keyword Mentions vs Oil Volatility')
        plt.show()

else:
    print('No oil volatility column found in daily_merged.')


### Added Explanation for Report / Presentation

To strengthen the project, additional keyword categories were added to capture more oil-specific, energy-market, macroeconomic, and geopolitical terms. This improves the ability of the model to distinguish general political posts from tweets that are more likely to be relevant to oil market movements.

The visualization section was also improved by directly comparing negative and positive tweet counts with next-day oil returns, and by comparing market/oil keyword mentions with oil volatility proxies. These plots make the relationship between tweet sentiment and oil behavior easier to interpret.

In addition, an event-study approach was added. Instead of relying only on simple correlation, the analysis identifies extreme positive and negative sentiment days and examines oil returns around those events. This provides stronger analytical value, although it should still be interpreted carefully as association rather than proven causality.

Finally, the model evaluation was expanded beyond accuracy by including balanced accuracy, precision, recall, F1 score, and ROC AUC. This gives a more honest view of model performance, especially because financial direction prediction can be imbalanced and noisy.


## 29) Feature Importance for Tree-Based Sentiment Model
Feature importance is not causality. It only shows what the model used for prediction.

In [ ]:
# Choose the best tree-based sentiment classifier if available.
tree_model_name = None
for candidate in ['Sentiment RF', 'Sentiment GB']:
    if candidate in classification_models:
        tree_model_name = candidate
        break

if tree_model_name is not None:
    feature_type, tree_model = classification_models[tree_model_name]
    tree_model.fit(Xs_train_scaled, y_clf_train)

    if hasattr(tree_model, 'feature_importances_'):
        importance_df = pd.DataFrame({
            'Feature': sentiment_features,
            'Importance': tree_model.feature_importances_
        }).sort_values('Importance', ascending=False).head(25)

        display(importance_df)

        if PLOTLY_OK:
            fig = px.bar(importance_df, x='Importance', y='Feature', orientation='h', title=f'Top Feature Importances — {tree_model_name}')
            style_fig(fig, height=700).show()

,Feature,Importance
250,total_market_importance_roll_mean_14,0.013489
31,oil_return_roll_mean_5,0.011374
215,avg_sentiment_roll_sum_7,0.011354
160,total_engagement_lag5,0.010946
235,avg_sentiment_roll_sum_14,0.010708
249,total_final_impact_roll_sum_14,0.010668
234,avg_sentiment_roll_mean_14,0.010184
56,avg_word_count,0.009464
175,total_engagement_lag7,0.009385
214,avg_sentiment_roll_mean_7,0.009280


## 30) Prediction Timeline for Best Classifier
This helps visually inspect when the model gets direction right or wrong.

In [ ]:
test_predictions_df = test_df[['day', 'target_next_day', 'target_next_day_return_pct', 'target_next_day_direction']].copy()
test_predictions_df['predicted_direction'] = best_clf_pred
test_predictions_df['correct'] = (test_predictions_df['predicted_direction'] == test_predictions_df['target_next_day_direction']).astype(int)

display(test_predictions_df.head(20))

if PLOTLY_OK:
    plot_df = test_predictions_df.copy()
    plot_df['Actual'] = plot_df['target_next_day_direction'].map({0: 'Down/Flat', 1: 'Up'})
    plot_df['Predicted'] = plot_df['predicted_direction'].map({0: 'Down/Flat', 1: 'Up'})
    fig = px.scatter(
        plot_df,
        x='day',
        y='target_next_day_return_pct',
        color='correct',
        hover_data=['Actual', 'Predicted'],
        title='Best Classifier: Correct vs Incorrect Predictions Over Time'
    )
    style_fig(fig, height=560).show()

,day,target_next_day,target_next_day_return_pct,target_next_day_direction,predicted_direction,correct
1536,2024-06-25 00:00:00+00:00,2024-06-26 00:00:00+00:00,0.222993,1,0,0
1537,2024-06-26 00:00:00+00:00,2024-06-27 00:00:00+00:00,1.088297,1,0,0
1538,2024-06-27 00:00:00+00:00,2024-06-28 00:00:00+00:00,-0.390911,0,0,1
1539,2024-06-28 00:00:00+00:00,2024-07-01 00:00:00+00:00,2.369552,1,0,0
1540,2024-07-01 00:00:00+00:00,2024-07-02 00:00:00+00:00,-0.743141,0,0,1
1541,2024-07-02 00:00:00+00:00,2024-07-03 00:00:00+00:00,0.817308,1,0,0
1542,2024-07-03 00:00:00+00:00,2024-07-05 00:00:00+00:00,-0.526310,0,0,1
1543,2024-07-05 00:00:00+00:00,2024-07-08 00:00:00+00:00,-0.986169,0,0,1
1544,2024-07-08 00:00:00+00:00,2024-07-09 00:00:00+00:00,-0.985159,0,1,0
1545,2024-07-09 00:00:00+00:00,2024-07-10 00:00:00+00:00,0.354481,1,0,0


## 31) Error Analysis by Market Activity
Does the classifier perform better on days with more market-related tweets?

In [ ]:
error_analysis_df = test_predictions_df.merge(
    daily_merged[['day', 'tweet_count', 'market_keyword_mentions', 'geo_keyword_mentions', 'oil_direct_keyword_mentions', 'market_related_posts']],
    on='day',
    how='left'
)

error_analysis_df['market_activity_bucket'] = pd.qcut(
    error_analysis_df['market_keyword_mentions'].rank(method='first'),
    q=3,
    labels=['Low Market Mentions', 'Medium Market Mentions', 'High Market Mentions']
)

bucket_perf = error_analysis_df.groupby('market_activity_bucket', observed=True).agg(
    rows=('correct', 'count'),
    accuracy=('correct', 'mean'),
    avg_market_mentions=('market_keyword_mentions', 'mean'),
    avg_oil_direct_mentions=('oil_direct_keyword_mentions', 'mean')
).reset_index()

display(bucket_perf)

if PLOTLY_OK:
    fig = px.bar(bucket_perf, x='market_activity_bucket', y='accuracy', text='accuracy', title='Prediction Accuracy by Market-Mention Activity')
    fig.update_traces(texttemplate='%{text:.3f}', textposition='outside')
    style_fig(fig, height=500).show()

,market_activity_bucket,rows,accuracy,avg_market_mentions,avg_oil_direct_mentions
0,Low Market Mentions,127,0.503937,0.834646,0.078740
1,Medium Market Mentions,127,0.566929,4.322835,0.582677
2,High Market Mentions,127,0.488189,14.393701,3.165354


## 32) Optional Hourly Analysis
This section is optional and descriptive. It should not be mixed with the daily model unless you carefully align timestamps.

Important fix: do not automatically choose Brent/WTI by row count if your daily target is WTI/CL=F. This notebook defaults to **WTI** when available.

In [ ]:
if oil_hourly_raw is None:
    print('No hourly oil file found. Skipping hourly analysis.')
else:
    oil_hourly = normalize_colnames(oil_hourly_raw)
    datetime_col = find_col(oil_hourly.columns, ['datetime', 'Date', 'date', 'time'])
    close_h_col = find_col(oil_hourly.columns, ['close', 'Close'])
    oil_type_col = find_col(oil_hourly.columns, ['oil_type', 'type', 'name', 'ticker'])

    if datetime_col is None or close_h_col is None:
        print('Hourly file found, but required datetime/close columns were not detected. Skipping hourly analysis.')
    else:
        oil_hourly = oil_hourly.rename(columns={datetime_col: 'datetime', close_h_col: 'hourly_close'})
        oil_hourly['datetime'] = pd.to_datetime(oil_hourly['datetime'], errors='coerce', utc=True)
        oil_hourly['hourly_close'] = pd.to_numeric(oil_hourly['hourly_close'], errors='coerce')
        oil_hourly = oil_hourly.dropna(subset=['datetime', 'hourly_close']).copy()

        if oil_type_col is not None:
            oil_hourly = oil_hourly.rename(columns={oil_type_col: 'oil_type'})
            available_types = oil_hourly['oil_type'].astype(str).str.upper().unique().tolist()
            print('Available hourly oil types:', available_types)

            if any('WTI' in t for t in available_types):
                chosen_type = [t for t in available_types if 'WTI' in t][0]
                oil_hourly_selected = oil_hourly[oil_hourly['oil_type'].astype(str).str.upper() == chosen_type].copy()
                print('Chosen hourly oil type:', chosen_type)
            else:
                chosen_type = oil_hourly['oil_type'].value_counts().idxmax()
                oil_hourly_selected = oil_hourly[oil_hourly['oil_type'] == chosen_type].copy()
                print('WTI not found. Chosen dominant oil type:', chosen_type)
        else:
            oil_hourly_selected = oil_hourly.copy()
            chosen_type = 'Unknown'

        oil_hourly_selected = oil_hourly_selected.sort_values('datetime')
        oil_hourly_selected['hourly_return_pct'] = oil_hourly_selected['hourly_close'].pct_change() * 100
        oil_hourly_selected['hour'] = oil_hourly_selected['datetime'].dt.floor('H')

        print('Hourly selected shape:', oil_hourly_selected.shape)
        display(oil_hourly_selected.head())

Available hourly oil types: ['BRENT', 'WTI']
Chosen hourly oil type: WTI
Hourly selected shape: (39837, 9)


,datetime,open,high,low,hourly_close,volume,oil_type,hourly_return_pct,hour
1,2017-01-20 00:00:00+00:00,52.30,52.31,52.24,52.29,0,WTI,NaN,2017-01-20 00:00:00+00:00
3,2017-01-20 01:00:00+00:00,52.28,52.28,52.14,52.16,0,WTI,-0.248614,2017-01-20 01:00:00+00:00
5,2017-01-20 02:00:00+00:00,52.15,52.35,52.12,52.35,0,WTI,0.364264,2017-01-20 02:00:00+00:00
7,2017-01-20 03:00:00+00:00,52.36,52.49,52.33,52.41,0,WTI,0.114613,2017-01-20 03:00:00+00:00
9,2017-01-20 04:00:00+00:00,52.42,52.79,52.30,52.74,0,WTI,0.629651,2017-01-20 04:00:00+00:00


## 33) Final Project Summary
This summary is generated from the actual model outputs.

In [ ]:
final_summary = []

final_summary.append(f'Dataset period: {daily_merged["day"].min()} to {daily_merged["day"].max()}')
final_summary.append(f'Sentiment engine used: {SENTIMENT_ENGINE}')
final_summary.append(f'Market-related posts detected: {int(df["is_market_related"].sum())}')
final_summary.append(f'Direct oil/energy posts detected: {int(df["is_oil_direct_related"].sum())}')
final_summary.append(f'Best regression model: {best_reg["Model"]} | R2={best_reg["R2"]:.4f} | MAE={best_reg["MAE"]:.4f}')
final_summary.append(f'Best classification model: {best_clf_name}')
final_summary.append(f'Best classification balanced accuracy: {clf_results_df.iloc[0]["Balanced Accuracy"]:.4f}')

if not baseline_only.empty and not sentiment_only.empty:
    final_summary.append(f'Sentiment improvement over best oil-only baseline balanced accuracy: {diff:.4f}')

print('========== FINAL PROJECT SUMMARY ==========')
for item in final_summary:
    print('-', item)

print('\nHonest interpretation:')
if best_reg['R2'] < 0:
    print('- Regression is weak; it does not reliably estimate next-day oil return size.')
else:
    print('- Regression has some signal, but should still be interpreted cautiously.')

if not baseline_only.empty and not sentiment_only.empty:
    if diff > 0.03:
        print('- Tweet/sentiment features add a meaningful directional improvement over oil-only baseline.')
    elif diff > 0:
        print('- Tweet/sentiment features add only a small directional improvement over oil-only baseline.')
    else:
        print('- Tweet/sentiment features do not beat the oil-only baseline.')

print('- This project is strongest as an EDA + weak-signal predictive experiment, not as proof that tweets control oil prices.')

========== FINAL PROJECT SUMMARY ==========
- Dataset period: 2017-02-09 00:00:00+00:00 to 2025-12-30 00:00:00+00:00
- Sentiment engine used: VADER
- Market-related posts detected: 6734
- Direct oil/energy posts detected: 1077
- Best regression model: Dummy Mean | R2=-0.0094 | MAE=1.4797
- Best classification model: Sentiment RF
- Best classification balanced accuracy: 0.5257
- Sentiment improvement over best oil-only baseline balanced accuracy: 0.0120

Honest interpretation:
- Regression is weak; it does not reliably estimate next-day oil return size.
- Tweet/sentiment features add only a small directional improvement over oil-only baseline.
- This project is strongest as an EDA + weak-signal predictive experiment, not as proof that tweets control oil prices.


## 34) Save Key Outputs
This exports the main tables for reporting.

In [ ]:
OUTPUT_DIR = Path('project_outputs')
OUTPUT_DIR.mkdir(exist_ok=True)

summary_cards.to_csv(OUTPUT_DIR / 'summary_cards.csv', index=False)
summary_keyword_check.to_csv(OUTPUT_DIR / 'keyword_detection_check.csv', index=False)
reg_results_df.to_csv(OUTPUT_DIR / 'regression_results.csv', index=False)
clf_results_df.to_csv(OUTPUT_DIR / 'classification_results.csv', index=False)
comparison_summary.to_csv(OUTPUT_DIR / 'baseline_comparison.csv', index=False)
corr_df.to_csv(OUTPUT_DIR / 'correlation_with_next_day_return.csv')
test_predictions_df.to_csv(OUTPUT_DIR / 'test_predictions.csv', index=False)

print('Saved outputs to:', OUTPUT_DIR.resolve())

Saved outputs to: /content/project_outputs


# End of Notebook

Recommended presentation wording:

> The model shows that tweet-based sentiment and market-signal features may contain a weak short-term directional signal for oil movement, but the improvement over oil-only baselines is small. Therefore, the results should be interpreted as evidence of limited association, not causal control or strong predictive power.